In [ ]:
#| default_exp rlm_lisette_v2

In [ ]:
#| export
import os
import json
import random
try:
    from litellm import register_model
except Exception:
    register_model = None  # litellm not available in CI; dependent code should handle this

## The plan

I want to implement recursive language models (https://alexzhang13.github.io/blog/2025/rlm/) with https://answerdotai.github.io/toolslm/shell.html as the python interpreter and `lisette` as the library to call LLMs. It already has great support for tool calling.

Here is my initial plan (just to get an overview).

We to be able to add the context to the python interpreter as the `context` variable. It is potentially very long. and we also need to define a function/tool in the repl which allows us to again call an LLM inside of the REPL.

And then we basically have an agentic loop which has the python interpreter with the preloaded `context` variable and the preloaded `call_lm()` function as a tool. And if I understand the `lisette` codebase (https://lisette.answer.ai/#tools) correctly, then I think that the normal chat function already has a `toolloop` kind of thing included.

I think `call_lm()` should basically just call a `lisette.client` under the hood.

Now that I think about it, it should not be that hard. But these are famous last words.

You can read urls with &`read_url`


Please read all urls above. Any holes in this plan?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠



<details class='tool-usage-details'>

```json
{
  "id": "toolu_01GBASPwZtBJGApsGK7Pvvu7",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://alexzhang13.github.io/blog/2025/rlm/"
    }
  },
  "result": "'[ Alex L. Zhang ](/) Toggle navigation\\n\\n  * [about ](/)\\n  * [blog ](/blog/)\\n  * [publications and pre-prints ](/publications/)\\n  * [projects ](/projects/)\\n  * [academics ](/academics/)\\n  * ctrl k __\\n  * ______\\n\\n# Recursive Language Models\\n\\nWe propose Recursive Language Models (RLMs), an inference strategy where language models can decompose and recursively interact with input context of unbounded length through REPL environments.\\n\\n## tl;dr\\n\\nWe explore language models that **recursively call themselves or other LLMs** before providing a final answer. Our goal is to enable the processing of essentially unbounded input context length and output length and to mitigate degradation \u201ccontext rot\u201d.\\n\\nWe propose **Recursive Language Models** , or **RLM** s, a general inference strategy where language models can decompose and recursively interact with their input context as a variable. We design a specific instantiation of this where GPT-5 or GPT-5-mini is queried in a Python REPL environment that stores the user\u2019s prompt in a variable.\\n\\nWe demonstrate that an **RLM using GPT-5-mini outperforms GPT-5** on a split of the most difficult long-context benchmark we got our hands on (OOLONG ) by more than **double** the number of correct answers, and is **cheaper** per query on average! We also construct a new long-context Deep Research task from BrowseComp-Plus . On it, we observe that RLMs outperform other methods like ReAct + test-time indexing and retrieval over the prompt. Surprisingly, we find that RLMs also do not degrade in performance when given 10M+ tokens at inference time.\\n\\nWe are excited to share these very early results, as well as argue that RLMs will be a powerful paradigm very soon. We think that RLMs trained explicitly to recursively reason are likely to represent the next milestone in **general-purpose inference-time scaling** after CoT-style reasoning models and ReAct-style agent models.\\n\\nWe have a compressed summary in the original tweet: <https://x.com/a1zhang/status/1978469116542337259>\\n\\nWe also now have a minimal implementation for people to build on top of: <https://github.com/alexzhang13/rlm>\\n\\n![Teaser Figure](/assets/img/rlm/teaser.png) **Figure 1.** An example of a recursive language model (RLM) call, which acts as a mapping from text \u2192 text, but is more flexible than a standard language model call and can scale to near-infinite context lengths. An RLM allows a language model to interact with an environment (in this instance, a REPL environment) that stores the (potentially huge) context, where it can recursively sub-query \u201citself\u201d, other LM calls, or other RLM calls, to efficiently parse this context and provide a final response.\\n\\n## Prelude: Why is \u201clong-context\u201d research so unsatisfactory?\\n\\nThere is this well-known but difficult to characterize phenomenon in language models (LMs) known as \u201ccontext rot\u201d. [Anthropic defines context rot](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents) as \u201c[when] the number of tokens in the context window increases, the model\u2019s ability to accurately recall information from that context decreases\u201d, but many researchers in the community know this definition doesn\u2019t _fully_ hit the mark. For example, if we look at popular needle-in-the-haystack benchmarks like [RULER](https://arxiv.org/abs/2404.06654), most frontier models actually do extremely well (90%+ on 1-year old models).\\n\\n![Pun kin](/assets/img/rlm/pumpkin.png) _I asked my LM to finish carving the pumpkin joke it started yesterday. It said, \u201cPumpkin? What pumpkin?\u201d \u2014 the context completely rotted._\\n\\nBut [people have noticed](https://x.com/kwindla/status/1962230672082497866) that context rot is this weird thing that happens when your Claude Code history gets bloated, or you chat with ChatGPT for a long time \u2014 it\u2019s almost like, as the conversation goes on, the model gets\u2026dumber? It\u2019s sort of this well-known but hard to describe failure mode that we don\u2019t talk about in our papers because we can\u2019t benchmark it. The natural solution is something along the lines of, \u201cwell maybe if I split the context into two model calls, then combine them in a third model call, I\u2019d avoid this degradation issue\u201d. We take this intuition as the basis for a recursive language model.\\n\\n## **Recursive Language Models (RLMs).**\\n\\nA recursive language model is a thin wrapper around a LM that can spawn (recursive) LM calls for intermediate computation \u2014 from the perspective of the user or programmer, it is the same as a model call. In other words, you query a RLM as an \u201cAPI\u201d like you would a LM, i.e. `rlm.completion(messages)` is a direct replacement for `gpt5.completion(messages)`. We take a **context-centric view** rather than a **problem-centric view** of input decomposition. This framing retains the functional view that we want a system that can answer a particular **query** over some associated **context** :\\n\\n![API](/assets/img/rlm/api.png) **Figure 2.** A recursive language model call replaces a language model call. It provides the user the illusion of near infinite context, while under the hood a language model manages, partitions, and recursively calls itself or another LM over the context accordingly to avoid context rot.\\n\\nUnder the hood, a RLM provides only the **query** to the LM (which we call the **root LM** , or LM with depth=0), and allows this LM to interact with an **environment** , which stores the (potentially huge) **context**.\\n\\nWe choose the **environment** to be a loop where the LM can write to and read the output of cells of a Python REPL Notebook (similar to a Jupyter Notebook environment) that is pre-loaded with the **context** as a variable in memory. The **root LM** has the ability to call a recursive LM (or LM with depth=1) inside the REPL **environment** as if it were a function in code, allowing it to naturally peek at, partition, grep through, and launch recursive sub-queries over the **context**. **Figure 3** shows an example of how the RLM with a REPL **environment** produces a final answer.\\n\\n![API](/assets/img/rlm/repl.png) **Figure 3.** Our instantiation of the RLM framework provides the root LM the ability to analyze the context in a Python notebook environment, and launch recursive LM calls (depth=1) over any string stored in a variable. The LM interacts by outputting code blocks, and it receives a (truncated) version of the output in its context. When it is done, it outputs a final answer with `FINAL(\u2026)` tags or it can choose to use a string in the code execution environment with `FINAL_VAR(\u2026)`.\\n\\nWhen the **root LM** is confident it has an answer, it can either directly output the answer as `FINAL(answer)`, or it can build up an answer using the variables in its REPL environment, and return the string inside that answer as `FINAL_VAR(final_ans_var)`.\\n\\nThis setup yields several benefits that are visible in practice:\\n\\n  1. The context window of the root LM is rarely clogged \u2014 because it never directly sees the entire context, its input context grows slowly.\\n  2. The root LM has the flexibility to view subsets of the context, or naively recurse over chunks of it. For example, if the query is to find a needle-in-the-haystack fact or multi-hop fact, the root LM can use `regex` queries to roughly narrow the context, then launch recursive LM calls over this context. This is particularly useful for arbitrary long context inputs, where indexing a retriever is expensive on the fly!\\n  3. The context can, in theory, be any modality that can be loaded into memory. The root LM has full control to view and transform this data, as well as ask sub-queries to a recursive LM.\\n\\n**Relationship to test-time inference scaling.** We are particularly excited about this view of language models because it offers another axis of scaling test-time compute. The trajectory in which a language model chooses to interact with and recurse over its context is entirely learnable, and can be RL-ified in the same way that reasoning is currently trained for frontier models. Interestingly, it does not directly require training models that can handle huge context lengths because **no single language model call should require handling a huge context**.\\n\\n**RLMs with REPL environments are powerful.** We highlight that the choice of the **environment** is flexible and not fixed to a REPL or code environment, but we argue that it is a good choice. The two key design choices of recursive language models are 1) treating the prompt as a Python variable, which can be processed programmatically in arbitrary REPL flows. This allows the LLM to figure out what to peek at from the long context, at test time, and to scale any decisions it wants to take (e.g., come up with its own scheme for chunking and recursion adaptively) and 2) allowing that REPL environment to make calls back to the LLM (or a smaller LLM), facilitated by the decomposition and versatility from choice (1).\\n\\nWe were excited by the design of CodeAct, and reasoned that adding recursive model calls to this system could result in significantly stronger capabilities \u2014 after all, LM function calls are incredibly powerful. However, we argue that RLMs fundamentally view LM usage and code execution differently than prior works: the **context** here is an object to be understood by the model, and code execution and recursive LM calls are a means of understanding this context efficiently. Lastly, in our experiments we only consider a recursive depth of 1 \u2014 i.e. the root LM can only call LMs, not other RLMs. It is a relatively easy change to allow the REPL environment to call RLMs instead of LMs, but we felt that for most modern \u201clong context\u201d benchmarks, a recursive depth of 1 was sufficient to handle most problems. However, for future work and investigation into RLMs, enabling larger recursive depth will naturally lead to stronger and more interesting systems.\\n\\n**The formal definition (click to expand)** Consider a general setup of a language model $M$ receiving a query $q$ with some associated, potentially long context $C = {[c_1,c_2,\u2026,c_m]}$. The standard approach is to treat $M(q,C)$ like a black box function call, which takes a query and context and returns some `str` output. We retain this frame of view, but define a thin scaffold on top of the model to provide a more **expressive** and **interpretable** function call $RLM_M(q,C)$ with the same input and output spaces. Formally, a recursive language model $RLM_{M}(q, C)$ over an environment $\\\\mathcal{E}$ similarly receives a query $q$ and some associated, potentially long context $C = [c_1,c_2,\u2026,c_m]$ and returns some `str` output. The primary difference is that we provide the model a tool call $RLM_M(\\\\hat{q}, \\\\hat{C})$, which spawns an isolated sub-RLM instance using a new query $\\\\hat{q}$ and a transformed version of the context $\\\\hat{C}$ with its own isolated environment $\\\\hat{\\\\mathcal{E}}$; eventually, the final output of this recursive callee is fed back into the environment of the original caller. The environment $\\\\mathcal{E}$ abstractly determines the control flow of how the language model $M$ is prompted, queried, and handled to provide a final output. In this paper, we specifically explore the use of a Python REPL environment that stores the input context $C$ as a variable in memory. This specific choice of environment enables the language model to **peek at** , **partition** , **transform** , and **map** over the input context and use recursive LMs to answer sub-queries about this context. Unlike prior agentic methods that rigidly define these workflow patterns, RLMs defer these decisions entirely to the language model. Finally, we note that particular choices of environments $\\\\mathcal{E}$ are flexible and are a generalization of a base model call: the simplest possible environment $\\\\mathcal{E}_0$ queries the model $M$ with input query and context $q, C$ and returns the model output as the final answer.\\n\\n## Some early (and very exciting) results!\\n\\nWe\u2019ve been looking around for benchmarks that reflect natural long-context tasks, e.g. long multi-turn Claude Code sessions. We namely were looking to highlight two properties that limit modern frontier models: 1) the context rot phenomenon, where model performance degrades as a function of context length, and 2) the system-level limitations of handling an enormous context.\\n\\nWe found in practice that many long-context benchmarks offer contexts that are not really that long and which were already solvable by the latest generation (or two) of models. In fact, we found some where **models could often answer queries without the context**! We luckily quickly found two benchmarks where modern frontier LLMs struggle to perform well, but we are [actively seeking](https://x.com/lateinteraction/status/1976964409139642716) any other good benchmark recommendations to try.\\n\\n### **Exciting Result #1 \u2014 Dealing with Context Rot.**\\n\\nThe **OOLONG** benchmark is a challenging new benchmark that evaluates long-context reasoning tasks over fine-grained information in context. We were fortunate to have the (anonymous _but not affiliated with us_) authors share the dataset upon request to run our experiments on a split of this benchmark.\\n\\n**Setup.** The `trec_coarse` split consists of 6 different types of queries to answer distributional queries about a giant list of \u201cquestion\u201d entries. For example, one question looks like:\\n\\n`For the following question, only consider the subset of instances that are associated with user IDs 67144, 53321, 38876, 59219, 18145, 64957, 32617, 55177, 91019, 53985, 84171, 82372, 12053, 33813, 82982, 25063, 41219, 90374, 83707, 59594. Among instances associated with these users, how many data points should be classified as label \\'entity\\'? Give your final answer in the form \\'Answer: number\\'.`\\n\\nThe query is followed by ~3000 - 6000 rows of entries with associated user IDs (not necessarily unique) and instances that **are not explicitly labeled** (i.e. the model has to infer the labeling to answer). They look something like this:\\n\\n```\\nDate: Dec 12, 2022 || User: 63685 || Instance: How many years old is Benny Carter ?\\nDate: Dec 30, 2024 || User: 35875 || Instance: What war saw battles at Parrot \\'s Beak and Black Virgin ?\\nDate: Apr 13, 2024 || User: 80726 || Instance: What Metropolis landmark was first introduced in the Superman cartoons of the 1940 \\'s ?\\nDate: Feb 29, 2024 || User: 59320 || Instance: When was Calypso music invented?\\n...\\n\\n```\\n\\nThe score is computed as the number of queries answered correctly by the model, with the caveat that for numerical / counting problems, they use a continuous scoring metric. This benchmark is extremely hard for both frontier models and agents because they have to **semantically** map and associate thousands of pieces of information in a single query, and cannot compute things a-priori! We evaluate the following models / agents:\\n\\n  * **GPT-5.** Given the whole context and query, tell GPT-5 to provide an answer.\\n  * **GPT-5-mini.** Given the whole context and query, tell GPT-5-mini to provide an answer.\\n  * **RLM(GPT-5-mini).** Given the whole context and query, tell RLM(GPT-5-mini) to provide an answer. GPT-5-mini (root LM) can recursively call GPT-5-mini inside its REPL environment.\\n  * **RLM(GPT-5) without sub-calls.** Given the whole context and query, tell RLM(GPT) to provide an answer. GPT-5 (root LM) cannot recursively call GPT-5 inside its REPL environment. This is an ablation for the use of a REPL environment without recursion.\\n  * **ReAct w/ GPT-5 + BM25.** We chunk every lines into its own \u201cdocument\u201d, and gives a ReAct loop access to a BM25 retriever to return 10 lines per search request.\\n\\n**Results.** We focus explicitly on questions with contexts over 128k tokens (~100 queries), and we track both the performance on the benchmark, as well as the overall API cost of each query. In all of the following results (Figure **4a,b**), **the entire input fits in the context window of GPT-5 / GPT-5-mini** \u2014 i.e., incorrect predictions are never due to truncation or context window size limitations:\\n\\n![API](/assets/img/rlm/oolong-132k.png) **Figure 4a.** We report the overall score for each method on the `trec_coarse` dataset of the OOLONG benchmark for queries that have a context length of 132k tokens. We compare performance to GPT-5. RLM(GPT-5-mini) outperforms GPT-5 by over **34 points (~114% increase)** , and is nearly as cheap per query (we found that the median query is cheaper due to some outlier, expensive queries).\\n\\nIt turns out actually that **RLM(GPT-5-_mini_)** outperforms **GPT-5** and **GPT-5-mini** by **> 33%**\u2191 raw score (over double the performance) while maintaining roughly the same total model API cost as **GPT-5** per query! When ablating recursion, we find that RLM performance degrades by ~10%, likely due to many questions requiring the model to answer semantic questions about the data (e.g. label each question). We see in **Figure 4b** that these gains roughly transfer when we double the size of the context to ~263k tokens as well, although with some performance degradation!\\n\\n![API](/assets/img/rlm/oolong-256k.png) **Figure 4b.** We report the overall score for each method on the trec_coarse dataset of the OOLONG benchmark for queries that have a context length of 263k tokens, nearly the limit for GPT-5/GPT-5-mini. We compare performance to GPT-5. RLM(GPT-5-mini) outperforms GPT-5 by over **15 points (~49% increase)** , and is cheaper per query on average.\\n\\nNotably, the performance of **GPT-5-mini** drops while **GPT-5** does not, which indicates that context rot is more severe for GPT-5-mini. We additionally noticed that the performance drop for the RLM approaches occurs for **_counting_** problems, where it makes more errors when the context length increases \u2014 for **GPT-5** , it already got most of these questions incorrect in the 132k context case, which explains why its performance is roughly preserved. Finally, while the **ReAct + GPT-5 + BM25** baseline doesn\u2019t make much sense in this setting, we provide it to show retrieval is difficult here while **RLM** is the more appropriate method.\\n\\nGreat! So we\u2019re making huge progress in solving goal (1), where GPT-5 has _just_ enough context window to fit the 263k case. But what about goal (2), where we may have 1M, 10M, or even 100M tokens in context? _Can we still treat this like a single model call?_\\n\\n### **Exciting Result #2 \u2014 Ridiculously Large Contexts**\\n\\nMy advisor Omar is a [superstar in the world of information retrieval (IR)](https://arxiv.org/abs/2004.12832), so naturally we also wanted to explore whether RLMs scale properly when given thousands (or more!) of documents. OOLONG provides a giant block of text that is difficult to index and therefore difficult to compare to retrieval methods, so we looked into [DeepResearch](https://openai.com/index/introducing-deep-research/)-like benchmarks that evaluate answering queries over documents.\\n\\n**Retrieval over huge offline corpuses.** We initially were interested in [BrowseComp](https://openai.com/index/browsecomp/) , which evaluates agents on multi-hop, web-search queries, where agents have to find the relevant documents online. We later found the [BrowseComp-Plus](https://arxiv.org/abs/2508.06600) benchmark, which pre-downloads all possible relevant documents for all queries in the original benchmark, and just provides a list of ~100K documents (~5k words on average) where the answer to a query is scattered across this list. For benchmarking RLMs, this benchmark is perfect to see if we can just throw ridiculously large amount of context into a single `chat.completion(...)` RLM call instead of building an agent!\\n\\n**Setup.** We explore how scaling the # documents in context affects the performance of various common approaches to dealing with text corpuses, as well as RLMs. Que<TRUNCATED>"
}
```

</details>



<details class='tool-usage-details'>

```json
{
  "id": "toolu_01VmJH9ghVCTkLmApVAPm93h",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://answerdotai.github.io/toolslm/shell.html"
    }
  },
  "result": "'[ ](./index.html) [ toolslm ](./index.html)\\n\\n__\\n\\n  1. [shell source](./shell.html)\\n\\n[ ](./index.html)\\n\\n  * [ toolslm](./index.html)\\n\\n  * [ xml source](./xml.html)\\n\\n  * [ funccall source](./funccall.html)\\n\\n  * [ shell source](./shell.html)\\n\\n  * [ Download helpers](./download.html)\\n\\n  * [ Markdown Hierarchy Parser](./md_hier.html)\\n\\n## On this page\\n\\n  * TerminalInteractiveShell.run_cell\\n  * get_shell\\n\\n  * [__Report an issue](https://github.com/AnswerDotAI/toolslm/issues/new)\\n\\n## Other Formats\\n\\n  * [ __CommonMark](shell.html.md)\\n\\n# shell source\\n\\nExported source\\n\\n```\\nimport ast, time, signal, traceback\\nfrom fastcore.utils import *\\n```\\n\\n__\\n\\n[`get_shell`](https://AnswerDotAI.github.io/toolslm/shell.html#get_shell) is like [`python`](https://AnswerDotAI.github.io/toolslm/funccall.html#python), except it also maintains a stateful interpreter, rather than just running a single line of code. This is implemented using IPython, so that must be installed.\\n\\nExported source\\n\\n```\\nfrom IPython.terminal.interactiveshell import TerminalInteractiveShell\\nfrom IPython.utils.capture import capture_output\\n```\\n\\n __\\n\\n```\\ndef exception2str(ex:Exception)->str:\\n    \"Convert exception `ex` into a string\"\\n    return \\'\\'.join(traceback.format_exception(type(ex), ex, ex.__traceback__))\\n```\\n\\n__\\n\\n```\\ntry: print(1/0)\\nexcept Exception as e: print(exception2str(e))\\n```\\n\\n__\\n\\n```\\nTraceback (most recent call last):\\n  File \"/var/folders/ss/34z569j921v58v8n1n_8z7h40000gn/T/ipykernel_37260/4058275565.py\", line 1, in <module>\\n    try: print(1/0)\\n               ~^~\\nZeroDivisionError: division by zero\\n\\n```\\n\\n* * *\\n\\n[source](https://github.com/AnswerDotAI/toolslm/blob/main/toolslm/shell.py#L19)\\n\\n### TerminalInteractiveShell.run_cell\\n\\n>\\n```\\n>      TerminalInteractiveShell.run_cell (cell, timeout=None)\\n```\\n\\n_Wrapper for original`run_cell` which adds timeout and output capture_\\n\\nExported source\\n\\n```\\nTerminalInteractiveShell.orig_run = TerminalInteractiveShell.run_cell\\n```\\n\\n __\\n\\nExported source\\n\\n```\\n@patch\\ndef run_cell(self:TerminalInteractiveShell, cell, timeout=None):\\n    \"Wrapper for original `run_cell` which adds timeout and output capture\"\\n    if timeout:\\n        def handler(*args): raise TimeoutError()\\n        signal.signal(signal.SIGALRM, handler)\\n        signal.alarm(timeout)\\n    try:\\n        with capture_output() as io: result = self.orig_run(cell)\\n        result.stdout = io.stdout\\n        return result\\n    except TimeoutException as e:\\n        result = self.ExecutionResult(error_before_exec=None, error_in_exec=e)\\n    finally:\\n        if timeout: signal.alarm(0)\\n```\\n\\n__\\n\\n* * *\\n\\n[source](https://github.com/AnswerDotAI/toolslm/blob/main/toolslm/shell.py#L35)\\n\\n### get_shell\\n\\n>\\n```\\n>      get_shell ()\\n```\\n\\n_Get a`TerminalInteractiveShell` with minimal functionality_\\n\\nExported source\\n\\n```\\ndef get_shell()->TerminalInteractiveShell:\\n    \"Get a `TerminalInteractiveShell` with minimal functionality\"\\n    sh = TerminalInteractiveShell()\\n    sh.logger.log_output = sh.history_manager.enabled = False\\n    dh = sh.displayhook\\n    dh.finish_displayhook = dh.write_output_prompt = dh.start_displayhook = lambda: None\\n    dh.write_format_data = lambda format_dict, md_dict=None: None\\n    sh.logstart = sh.automagic = sh.autoindent = False\\n    sh.autocall = 0\\n    sh.system = lambda cmd: None\\n    return sh\\n```\\n\\n __\\n\\n```\\nshell = get_shell()\\n```\\n\\n__\\n\\n```\\nr = shell.run_cell(\\'print(3); 1+1\\')\\nr.result,r.stdout\\n```\\n\\n __\\n\\n```\\n(2, \\'3\\\\n\\')\\n```\\n\\n```\\nr = shell.run_cell(\\'raise Exception(\"blah\")\\')\\nprint(exception2str(r.error_in_exec))\\n```\\n\\n__\\n\\n```\\nTraceback (most recent call last):\\n  File \"/Users/jhoward/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py\", line 3577, in run_code\\n    exec(code_obj, self.user_global_ns, self.user_ns)\\n  File \"<ipython-input-1-338156281413>\", line 1, in <module>\\n    raise Exception(\"blah\")\\nException: blah\\n\\n```\\n\\n```\\nr = shell.run_cell(\\'import time; time.sleep(10)\\', timeout=1)\\nr.error_in_exec\\n```\\n\\n __\\n\\n```\\nTimeoutError()\\n```\\n\\n  * [__Report an issue](https://github.com/AnswerDotAI/toolslm/issues/new)'"
}
```

</details>



<details class='tool-usage-details'>

```json
{
  "id": "toolu_01FaCKJNR86853D7P6dvuc5H",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://lisette.answer.ai/#tools"
    }
  },
  "result": "'## Tools\\n\\nLisette makes it easy to give LLMs access to Python functions. Just define a function with type hints and a docstring:\\n\\n```\\ndef add_numbers(\\n    a: int,  # First number to add\\n    b: int   # Second number to add  \\n) -> int:\\n    \"Add two numbers together\"\\n    return a + b\\n```\\n\\n __\\n\\nNow pass the function to [`Chat`](https://lisette.answer.ai/core.html#chat) and the model can use it automatically:\\n\\n```\\nchat = Chat(models[0], tools=[add_numbers])\\nres = chat(\"What\\'s 47 + 23? Use the tool.\")\\nres\\n```\\n\\n __\\n\\nThe result of 47 + 23 is 70.\\n\\n  * id: `chatcmpl-xxx`\\n  * model: `claude-sonnet-4-20250514`\\n  * finish_reason: `stop`\\n  * usage: `Usage(completion_tokens=18, prompt_tokens=573, total_tokens=591, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`\\n\\nIf you want to see all intermediate messages and outputs you can use the `return_all=True` feature.\\n\\n```\\nchat = Chat(models[0], tools=[add_numbers])\\nres = chat(\"What\\'s 47 + 23 + 59? Use the tool.\",max_steps=3,return_all=True)\\ndisplay(*res)\\n```\\n\\n__\\n\\nI\u2019ll help you calculate 47 + 23 + 59 using the add_numbers tool. Since the tool can only add two numbers at a time, I\u2019ll need to do this in two steps.\\n\\n\ud83d\udd27 add_numbers({\u201ca\u201d: 47, \u201cb\u201d: 23})\\n\\n  * id: `chatcmpl-xxx`\\n  * model: `claude-sonnet-4-20250514`\\n  * finish_reason: `tool_calls`\\n  * usage: `Usage(completion_tokens=116, prompt_tokens=433, total_tokens=549, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`\\n\\n```\\n{\\'tool_call_id\\': \\'toolu_01F9oakoP8ANHkTMD1DyQDi7\\',\\n \\'role\\': \\'tool\\',\\n \\'name\\': \\'add_numbers\\',\\n \\'content\\': \\'70\\'}\\n```\\n\\nNow I\u2019ll add the result (70) to the third number (59):\\n\\n\ud83d\udd27 add_numbers({\u201ca\u201d: 70, \u201cb\u201d: 59})\\n\\n  * id: `chatcmpl-xxx`\\n  * model: `claude-sonnet-4-20250514`\\n  * finish_reason: `tool_calls`\\n  * usage: `Usage(completion_tokens=87, prompt_tokens=562, total_tokens=649, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`\\n\\n```\\n{\\'tool_call_id\\': \\'toolu_01Cdf3FHJdbx64F8H8ooE1Db\\',\\n \\'role\\': \\'tool\\',\\n \\'name\\': \\'add_numbers\\',\\n \\'content\\': \\'129\\'}\\n```\\n\\nThe answer is **129**.\\n\\nI calculated this by first adding 47 + 23 = 70, then adding 70 + 59 = 129.\\n\\n  * id: `chatcmpl-xxx`\\n  * model: `claude-sonnet-4-20250514`\\n  * finish_reason: `stop`\\n  * usage: `Usage(completion_tokens=41, prompt_tokens=702, total_tokens=743, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None), cache_creation_input_tokens=0, cache_read_input_tokens=0)`\\n\\nIt shows the intermediate tool calls, and the tool results!'"
}
```

</details>

Thanks for sharing these URLs! I've read through them. Your plan looks pretty solid overall. Here are a few things to think about:

**Potential considerations:**

1. **Context size handling** - The RLM paper shows contexts can be huge (10M+ tokens). How will you handle passing very large contexts into the Python REPL? Will you need streaming/chunking strategies?

2. **Tool calling depth** - The paper mentions they only use recursive depth of 1 (root LM calls LMs, not other RLMs). Do you want to support deeper recursion? If so, you'll need to think about how to prevent infinite loops and manage the call stack.

3. **Truncation strategy** - toolslm's shell captures output, but what happens if the recursive `call_lm()` returns a very long response? Will you truncate it before returning to the root LM?

4. **Timeout handling** - The shell already has timeout support. Should `call_lm()` also have its own timeout to prevent hanging on slow API calls?

5. **System prompt for the root LM** - What instructions will you give the root LM about how to use the REPL and `call_lm()`? The paper mentions specific prompting strategies.

Do any of these concerns resonate with your use case, or do you have thoughts on how you'd handle them?

re Context length: that is the beauty if this approach and exactly what makes this work. Since the context only lives in the python REPL it can be super long since the LLM can use python to inspect the context, chunk it, search in it with grep etc. There is virtually no restriction on how long a string can be in a python environment (if you have context that is bigger than your RAM, then you have that problem, yes :D).

For the system prompt we will use prompts similar to the ones above I got from the "official repo".

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Ah, you're absolutely right! I was thinking about it wrong. The context lives in the REPL's memory, not in the LLM's context window. That's elegant.

Looking at your system prompt from the repo, it seems comprehensive. A few quick observations:

1. **The prompt is quite long** - Have you tested whether models actually follow all those instructions, or do they sometimes skip steps (like not checking context before answering)?

2. **The "safeguard" on iteration 0** is nice - forces the model to look at context first.

3. **The recursion strategy** - The prompt encourages chunking and recursive calls, which aligns with the paper's approach.

One thing I notice: the prompt says sub-LLMs can handle "around 500K chars". Is that hardcoded, or will you make it configurable based on which model you're using for recursive calls?

Also, are you planning to use the same model for root and recursive calls, or mix models (like the paper does with GPT-5 calling GPT-5-mini)?

In [ ]:
#| export
from lisette import *

In [ ]:
#| eval: false
?Chat

```python
def Chat(
    model:str, # LiteLLM compatible model name
    sp:str='', # System prompt
    temp:int=0, # Temperature
    search:bool=False, # Search (l,m,h), if model supports it
    tools:list=None, # Add tools
    hist:list=None, # Chat history
    ns:Optional=None, # Custom namespace for tool calling
    cache:bool=False, # Anthropic prompt caching
    cache_idxs:list=[-1], # Anthropic cache breakpoint idxs, use `0` for sys prompt if provided
    ttl:NoneType=None, # Anthropic prompt caching ttl
    api_base:NoneType=None, # API base URL for custom providers
    api_key:NoneType=None, # API key for custom providers
    extra_headers:NoneType=None, # Extra HTTP headers for custom providers
    tc_refs:bool=False, # Enable tool call result references
    tc_res_eval:bool=False, # literal_eval tool results before storing in tc_res
    markup:int=0, # Cost markup multiplier (e.g. 0.5 for 50%)
):

```



```
<no docstring>
```



**File:** `/usr/local/lib/python3.12/site-packages/lisette/core.py`

**Type:** type

In [ ]:
#| eval: false
import os
c = Chat(model="openai/gpt-oss-20b", api_base="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

c("who are you?", max_tokens=1000)

I’m ChatGPT, a large language model created by OpenAI. I’m trained on a wide range of text and can help answer questions, brainstorm ideas, explain concepts, draft emails, write code, and much more. Think of me as a virtual assistant that’s here to chat and assist you with whatever you need!

<details>

- id: `gen-1774860272-PFKLpWT59vXYvpWMMrJO`
- model: `openai/gpt-oss-20b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=125, prompt_tokens=71, total_tokens=196, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=58, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=2.5695e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.5695e-05, 'upstream_inference_prompt_cost': 3.195e-06, 'upstream_inference_completions_cost': 2.25e-05})`

</details>

## The python sandbox

In [ ]:
#| export
from toolslm.shell import get_shell

In [ ]:
#| eval: false
sh = get_shell()

In [ ]:
#| eval: false
sh.run_cell("15 + 35")

<ExecutionResult object at 74c0ef7d7b30, execution_count=1 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 74c0ef7d7560, raw_cell="15 + 35" transformed_cell="15 + 35
" store_history=False silent=False shell_futures=True cell_id=None> result=50>

In [ ]:
#| eval: false
bp = """---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tinygrad helped me finally understand what Sutton meant
---

Have you heard about The Bitter Lesson? If you are on Twitter and follow some ML/AI people, I bet you have.

If you haven't: It is a [blog post by Richard Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html), a Canadian computer scientist, who wrote one of *the* books about reinforcement learning.

Sutton's core argument is simple: general methods that leverage search and compute outperform domain-specific solutions. By "search," he doesn't mean information retrieval - he means using learning algorithms to explore possible solutions or strategies. His go-to example is computer chess: early engines relied heavily on chess expertise to evaluate positions, while AlphaZero knew only the basic rules and learned by playing itself, eventually surpassing any human player.

This lesson is "bitter" because it suggests our clever insights don't matter. Our domain-specific knowledge gets crushed by what looks like "dumb brute-force search and compute." If you're an engineer, this naturally raises an uncomfortable question: what's the point of us?

## The Click

For a long time, I didn't really understand Sutton's point. What does it mean to solve problems using search and compute? What is the role of engineers then? Are we just supposed to prompt a LLM and hope for the best? That can't scale.

My aha moment came from an unlikely source: [`tinygrad`](https://github.com/tinygrad/tinygrad). It's a Python library similar to PyTorch but much simpler on the surface. I think George Hotz initially built it as a toy project after watching Andrej Karpathy create micrograd, but it evolved into a solid tensor library that runs LLMs efficiently on different hardware through clever optimizations.

George's blog post ["Can tinygrad win?"](https://geohot.github.io//blog/jekyll/update/2025/07/06/can-tinygrad-win.html) made everything click. He lays out his vision for the library: the entire stack of making code run fast on hardware is fundamentally a search problem across multiple layers — searching for better scheduling algorithms, faster GPU kernels, all of it. The task of the library is to "Expose the underlying search problem spanning several orders of magnitude." and then "Apply the state of the art in search. Burn compute.".

That's when I finally understood what Sutton meant. Like in computer chess, raw compute and effective search will beat manually crafted solutions.

But the key insight is this: someone still has to expose that search problem. Someone has to frame it correctly.

In this post, I'll show you why our work as engineers still matters, just differently. The Bitter Lesson isn't bitter at all - it's a blueprint for better engineering. And the pattern of search & compute appears everywhere once you know what to look for.

## The Framework

For me, The Bitter Lesson is all about building effective search systems. Again, not search in the sense of information retrieval, but in the sense of allowing raw compute to explore a space of solutions effectively.

For that we need three core components:
1. An effective problem formulation that exposes the right parameters in a way that different algorithms can dig into it.
2. A way to evaluate possible solutions (evals) and provide feedback to the search mechanism.
3. A way to define constraints that a solution has to satisfy (memory, time, specific form etc.)

Our job as engineers is to build these three components. We design the problem-specific, scalable infrastructure that enables search to work. Then we gtfo and let compute do what it does best.

But don't you worry, there is still place for your precious domain knowledge, even in The Bitter Lesson, but you should use it elsewhere:

Use your domain knowledge to:

* Design problem representations (make search tractable)
* Build evaluations (capture what matters)
* Set constraints (avoid unsafe regions)
* Initialize search (warm start)

And use search to:

* Explore solution space
* Find non-obvious patterns
* Exceed human intuition
* Scale the solution with compute

Sound familiar? We've done this before — moving from punch cards to text-based code was the same kind of shift. And now we're building harnesses and guardrails (and [RL environments](https://www.primeintellect.ai/blog/environments)) that define the bounds of desired solutions rather than crafting the solutions directly.

## The Examples

It's fascinating to see where these themes come up, besides `tinygrad`.

If you follow me on [Twitter](https://twitter.com/rasmus1610), you might know that I'm a huge fanboi of DSPy - a declarative library for "programming not prompting" LLMs. And DSPy embodies the approach described above by separating problem definition from solution strategy. Omar Khattab, the creator of the library, actually gave a talk called ["On Engineering AI Systems that Endure The Bitter Lesson"](https://youtu.be/qdmxApz3EJI?si=P8CFiY1gD1lCMFoG) at this year's AI Engineering conference in San Francisco. Interestingly, his vision of DSPy mirrors George Hotz's vision for tinygrad: provide the right level of abstraction, then let search & compute do its thing.

One key mechanism here is [DSPy's signatures](https://dspy.ai/learn/programming/signatures/). A DSPy signature is like a function signature in the sense that you specify your inputs and the shape of your desired outputs. It gives you the flexibility to go from input to output however you want. Sure, currently this "however" means LLMs, but just because they are the most flexible piece of computing we currently have. But if some other paradigm (or even a newer LLM) comes around the corner, your DSPy modules are very easy to adapt to it.

This search & compute principle is also captured beautifully in a talk I stumbled upon some weeks ago. [Rens Dimmendaal](https://rensdimmendaal.com/) on the [Solveit Discord server](https://solve.it.com) pointed me to [a talk by Raymond Hettinger](https://www.youtube.com/watch?v=_GP9OpZPUYc) aptly titled "Modern solvers: Problems well-defined are problems solved". Just the title captures perfectly what I think The Bitter Lesson is all about.

One of the topics in his talk, SAT solvers, shows how problem formulation may become the critical engineering work. For those who don't know what a [SAT solver](https://en.wikipedia.org/wiki/SAT_solver) is, go and read the Wikipedia page. I'm kidding. It's a piece of software which can solve boolean algebra problems very effectively - and you would be surprised how many problems can be formulated as boolean algebra problems. But - without going into too much technical detail (you can really read the wikipedia page - it's linked above) - it expects the input to be in a very specific logical form, called the [conjunctive normal form (CNF)](https://en.wikipedia.org/wiki/Conjunctive_normal_form). And it turns out that just defining your problem in CNF is often surprisingly tricky. But once you got there, a modern SAT solver gives you the solution very fast, even for very complex problems.

It's also no coincidence that [AI evals](https://maven.com/parlance-labs/evals) have become such a hot topic for building LLM apps recently (h/t [Hamel Husain](https://hamel.dev)). When you're iterating on LLM applications - whether manually or with prompt optimizers - you're essentially searching through a solution space. But unlike a binary classifier where you can rely on simple metrics like accuracy, LLMs have massive output spaces that are much harder to evaluate. Here, building robust evals becomes critical engineering work: you need to define what "good" means for your specific problem, build systems to measure it reliably, and scale that evaluation. This is another instance of building the harness — the infrastructure that lets search & compute work effectively.

But this principle even extends to the physical world. [Proxima Fusion](https://www.youtube.com/watch?v=3Nr__6DDk6I) optimized their stellarator fusion reactor design through numerical simulation—defining the physics constraints, then letting compute search the design space before building the hardware. [AlphaFold](https://alphafold.ebi.ac.uk/) does something similar for proteins: define the structure prediction problem, set up physical constraints, then let search explore conformational space.

## The Conclusion

Look, maybe I'm fooling myself here. Maybe Sutton is right all the way down, and eventually even problem formulation gets automated. Maybe we'll all end up sipping synthetic mojitos in the metaverse while AGI does everything. But we're clearly not there yet - and I don't think we will be anytime soon.

So for now, when I think of The Bitter Lesson, I'm not depressed because we software engineers will become obsolete, but I'm excited because it points to what we should focus on: Building scalable systems that leverage search and compute. It's about moving up an abstraction level, building harnesses instead of solutions and defining problems instead of hardcoding solutions (hardcoding is rarely a good thing it seems).

And in my mind, that sounds just as interesting."""

In [ ]:
#| eval: false
sh = get_shell()
sh.user_ns['context'] = bp

In [ ]:
#| eval: false
sh.run_cell(f"context={bp}")

<ExecutionResult object at 74c0ef849880, execution_count=1 error_before_exec=invalid decimal literal (<ipython-input-1-bbd02197e81e>, line 3) error_in_exec=None info=<ExecutionInfo object at 74c0ef849850, raw_cell="context=---
title: "The Not-so Bitter Lesson"
date.." transformed_cell="context=---
title: "The Not-so Bitter Lesson"
date.." store_history=False silent=False shell_futures=True cell_id=None> result=None>

Oh I guess we have to "sanitize" or at least escape the context string?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, exactly! The issue is that the blog post contains characters that Python is interpreting as code syntax (like the dashes in the YAML frontmatter).

You have a few options:

1. **Use triple quotes** - Wrap the string in `"""` or `'''`:
```python
sh.run_cell(f'context = """{bp}"""')
```

2. **Use repr()** - This automatically escapes special characters:
```python
sh.run_cell(f'context = {repr(bp)}')
```

3. **Pass it directly to the namespace** - Instead of running code, inject it:
```python
sh.user_ns['context'] = bp
```

The third option is cleanest for your RLM use case since you're not actually executing user code - you're just setting up the environment. Want to try one of these?

In [ ]:
#| eval: false
sh = get_shell()
sh.user_ns['context'] = bp

Explain what's going on in the previous code cell?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The code is doing two things:

1. **`sh = get_shell()`** - Creates a fresh Python REPL (Read-Eval-Print-Loop) environment. This is like starting a new Python interpreter session.

2. **`sh.user_ns['context'] = bp`** - Directly injects the blog post string into the REPL's namespace as a variable called `context`. The `.user_ns` is a dictionary that holds all variables in that Python session.

This approach bypasses running Python code to set the variable (which would require escaping special characters). Instead, it directly places the value into the interpreter's memory.

For your RLM implementation, this is exactly what you want - you're pre-loading the `context` variable so the LLM can access it when it runs code in the REPL.

Does this make sense? Want to see what happens when you try to access `context` from within the shell?

In [ ]:
#| eval: false
sh.run_cell("context[:200]")

<ExecutionResult object at 74c0ef72cf20, execution_count=1 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 74c0ef72cad0, raw_cell="context[:200]" transformed_cell="context[:200]
" store_history=False silent=False shell_futures=True cell_id=None> result='---\ntitle: "The Not-so Bitter Lesson"\ndate: 2025-10-20T15:27:52+01:00\ndraft: false\nexcerpt: How tinygrad helped me finally understand what Sutton meant\n---\n\nHave you heard about The Bitter Lesson? If '>

better.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great! So now you have the context loaded into the REPL environment.

Next step: do you want to implement the `llm_query()` function that the root LM can call recursively? This would be a function in the shell namespace that takes a string and calls your lisette client.

Want to sketch out what that function should look like, or would you like me to suggest an approach?

Grab the output from the LLM call only.

would setting the max_token limit to a range be feasible. I think it is enforced by lisette

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Looking at the `Chat` signature from earlier, `max_tokens` is passed through to the underlying LiteLLM call, which does enforce it. So yes, setting a limit is feasible and probably a good idea.

A few considerations:

1. **What's a reasonable default?** The RLM paper mentions sub-LLMs can handle ~500K chars. Depending on your model, that might be 100K-200K tokens. But for outputs, you probably want something smaller to avoid huge responses clogging the root LM's context.

2. **Should it be configurable per call?** The root LM might want shorter answers for simple queries vs longer ones for complex summarization.

3. **What happens if the response gets truncated?** Should the function indicate that somehow, or just return whatever it got?

What range were you thinking?

In [ ]:
#| eval: false
# Demo call removed to keep notebook tests order-safe.

'The capital of France is **Paris**.'

This works, but we don't need a stateful chat class for a onetime interaction, so lets refactor that to use `completion`

In [ ]:
#| export
import litellm

In [ ]:
#| eval: false
mk_msg("What is the capital of france?")

{'role': 'user', 'content': 'What is the capital of france?'}

In [ ]:
#| eval: false
?mk_msg

```python
def mk_msg(
    content, # Content: str, bytes (image), list of mixed content, or dict w 'role' and 'content' fields
    role:str='user', # Message role if content isn't already a dict/Message
    cache:bool=False, # Enable Anthropic caching
    ttl:NoneType=None, # Cache TTL: '5m' (default) or '1h'
):

```



```
Create a LiteLLM compatible message.
```



**File:** `/usr/local/lib/python3.12/site-packages/lisette/core.py`

**Type:** function

In [ ]:
#| eval: false
?litellm.completion

```python
def completion(
    model:str, # type: ignore # noqa: PLR0915
    messages:List=[], # Optional OpenAI params: see https://platform.openai.com/docs/api-reference/chat/create
    timeout:Union=None, temperature:Optional=None, top_p:Optional=None, n:Optional=None, stream:Optional=None,
    stream_options:Optional=None, stop:NoneType=None, max_completion_tokens:Optional=None, max_tokens:Optional=None,
    modalities:Optional=None, prediction:Optional=None, audio:Optional=None, presence_penalty:Optional=None,
    frequency_penalty:Optional=None, logit_bias:Optional=None, user:Optional=None,
    reasoning_effort:Optional=None, # openai v1.0+ new params
    verbosity:Optional=None, response_format:Union=None, seed:Optional=None, tools:Optional=None,
    tool_choice:Union=None, logprobs:Optional=None, top_logprobs:Optional=None, parallel_tool_calls:Optional=None,
    web_search_options:Optional=None, deployment_id:NoneType=None, extra_headers:Optional=None,
    safety_identifier:Optional=None, service_tier:Optional=None,
    functions:Optional=None, # soon to be deprecated params by OpenAI
    function_call:Optional=None, base_url:Optional=None, # set api_base, api_version, api_key
    api_version:Optional=None, api_key:Optional=None,
    model_list:Optional=None, # pass in a list of api_base,keys, etc.
    thinking:Optional=None, # Optional liteLLM function params
    shared_session:Optional=None, # Session management
    enable_json_schema_validation:Optional=None, # Per-request JSON schema validation (overrides litellm.enable_json_schema_validation)
    kwargs:VAR_KEYWORD
)->Union:

```



```
Perform a completion() using any of litellm supported llms (example gpt-4, gpt-3.5-turbo, claude-2, command-nightly)
Parameters:
    model (str): The name of the language model to use for text completion. see all supported LLMs: https://docs.litellm.ai/docs/providers/
    messages (List): A list of message objects representing the conversation context (default is an empty list).

    OPTIONAL PARAMS
    functions (List, optional): A list of functions to apply to the conversation messages (default is an empty list).
    function_call (str, optional): The name of the function to call within the conversation (default is an empty string).
    temperature (float, optional): The temperature parameter for controlling the randomness of the output (default is 1.0).
    top_p (float, optional): The top-p parameter for nucleus sampling (default is 1.0).
    n (int, optional): The number of completions to generate (default is 1).
    stream (bool, optional): If True, return a streaming response (default is False).
    stream_options (dict, optional): A dictionary containing options for the streaming response. Only set this when you set stream: true.
    stop(string/list, optional): - Up to 4 sequences where the LLM API will stop generating further tokens.
    max_tokens (integer, optional): The maximum number of tokens in the generated completion (default is infinity).
    max_completion_tokens (integer, optional): An upper bound for the number of tokens that can be generated for a completion, including visible output tokens and reasoning tokens.
    modalities (List[ChatCompletionModality], optional): Output types that you would like the model to generate for this request.. You can use `["text", "audio"]`
    prediction (ChatCompletionPredictionContentParam, optional): Configuration for a Predicted Output, which can greatly improve response times when large parts of the model response are known ahead of time. This is most common when you are regenerating a file with only minor changes to most of the content.
    audio (ChatCompletionAudioParam, optional): Parameters for audio output. Required when audio output is requested with modalities: ["audio"]
    presence_penalty (float, optional): It is used to penalize new tokens based on their existence in the text so far.
    frequency_penalty: It is used to penalize new tokens based on their frequency in the text so far.
    logit_bias (dict, optional): Used to modify the probability of specific tokens appearing in the completion.
    user (str, optional):  A unique identifier representing your end-user. This can help the LLM provider to monitor and detect abuse.
    logprobs (bool, optional): Whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message
    top_logprobs (int, optional): An integer between 0 and 5 specifying the number of most likely tokens to return at each token position, each with an associated log probability. logprobs must be set to true if this parameter is used.
    metadata (dict, optional): Pass in additional metadata to tag your completion calls - eg. prompt version, details, etc.
    api_base (str, optional): Base URL for the API (default is None).
    api_version (str, optional): API version (default is None).
    api_key (str, optional): API key (default is None).
    model_list (list, optional): List of api base, version, keys
    extra_headers (dict, optional): Additional headers to include in the request.

    LITELLM Specific Params
    mock_response (str, optional): If provided, return a mock completion response for testing or debugging purposes (default is None).
    custom_llm_provider (str, optional): Used for Non-OpenAI LLMs, Example usage for bedrock, set model="amazon.titan-tg1-large" and custom_llm_provider="bedrock"
    max_retries (int, optional): The number of retries to attempt (default is 0).
Returns:
    ModelResponse: A response object containing the generated completion and associated metadata.

Note:
    - This function is used to perform completions() using the specified language model.
    - It supports various optional parameters for customizing the completion behavior.
    - If 'mock_response' is provided, a mock completion response is returned for testing or debugging.
```



**File:** `/usr/local/lib/python3.12/site-packages/litellm/main.py`

**Type:** function

In [ ]:
#| eval: false
litellm.completion("openai/gpt-oss-20b", [mk_msg("What is the capital of france?")], base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY"))

The capital of France is **Paris**.

<details>

- id: `gen-1774860275-AFQaDRf6W0dVR7MJwTPA`
- model: `openai/gpt-oss-20b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=41, prompt_tokens=76, total_tokens=117, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=23, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=9.19e-06, is_byok=False, cost_details={'upstream_inference_cost': 9.19e-06, 'upstream_inference_prompt_cost': 3.04e-06, 'upstream_inference_completions_cost': 6.15e-06})`

</details>

is there a way to choose a specific provider in openrouter?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! OpenRouter lets you specify providers in two ways:

1. **In the model name** - Add the provider prefix: `"openrouter/anthropic/claude-sonnet-4"` or `"openrouter/openai/gpt-4"`

2. **Via `extra_headers`** - Pass a provider preference:
   ```python
   extra_headers={"X-Provider-Preference": "Anthropic,OpenAI"}
   ```

The first method forces a specific provider, while the second provides a fallback order.

Want to try one of these approaches for your `llm_query()` function?

Now we can refactor our llm_query function

Can I add this to the user_ns like we did with the context variable?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes! You can add the function to the shell's namespace the same way:

```python
sh.user_ns['llm_query'] = llm_query
```

Then the root LM will be able to call it from within the REPL. Want to test it by running a cell that calls `llm_query()`?

In [ ]:
#| eval: false
sh.user_ns['llm_query'] = llm_query
sh.run_cell("llm_query('what is the capital of france?')")

<ExecutionResult object at 74c0ed5d6540, execution_count=1 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 74c0ed5d6600, raw_cell="llm_query('what is the capital of france?')" transformed_cell="llm_query('what is the capital of france?')
" store_history=False silent=False shell_futures=True cell_id=None> result=ModelResponse(id='gen-1774860279-wj7fDpcTsXCNvFrsAGiW', created=1774860279, model='openai/gpt-oss-120b', object='chat.completion', system_fingerprint=None, choices=[Choices(finish_reason='stop', index=0, message=Message(content='The capital of France is **Paris**.', role='assistant', tool_calls=None, function_call=None, reasoning_content='The user asks: "what is the capital of france?" Simple answer: Paris. Provide answer. Maybe some extra info.', provider_specific_fields={'refusal': None, 'reasoning_details': [{'type': 'reasoning.text', 'text': 'The user asks: "what is the capital of france?" Simple answer: Paris. Provide answer. Maybe some extra info.',

But we don't want to return a `ModelResponse` object but just the string, so we have to "unpack that"

In [ ]:
#| export
def llm_query(query: str, model="openai/gpt-oss-120b", 
              base_url="https://openrouter.ai/api/v1", 
              api_key=os.getenv("OPENROUTER_API_KEY")):
    msg = mk_msg(query)
    response = litellm.completion(model, [msg], base_url=base_url, api_key=api_key)
    return response.choices[0].message.content


'The capital of France is **Paris**.'

In [ ]:
#| eval: false
sh.user_ns['llm_query'] = llm_query
sh.run_cell("llm_query('what is the capital of france?')")

<ExecutionResult object at 74c0ed4b1ee0, execution_count=1 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 74c0ed4b20c0, raw_cell="llm_query('what is the capital of france?')" transformed_cell="llm_query('what is the capital of france?')
" store_history=False silent=False shell_futures=True cell_id=None> result='The capital of France is **Paris**.'>

Easy peasy. Now we just need to wrap the shell into a tool for the "outer" LLM to use, add a system prompt and we are done.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Exactly! You're almost there.

For the tool wrapper, you'll need a function that:
1. Takes code as input (the `repl` code blocks from the LLM)
2. Runs it via `sh.run_cell()`
3. Returns the result (probably with some truncation for long outputs)

Do you want to write that function, or would you like me to show you a quick example?

Show me a quick example

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's a quick example:

```python
def run_repl(code: str) -> str:
    "Execute Python code in the REPL environment"
    result = sh.run_cell(code)
    if result.error_in_exec:
        return f"Error: {result.error_in_exec}"
    output = result.stdout or ""
    if result.result is not None:
        output += str(result.result)
    return output[:5000]  # truncate long outputs
```

This captures both stdout (from `print()`) and the result (from expressions). The truncation prevents overwhelming the root LM's context.

Want to test it out, or adjust anything?

## `run_repl`

In [ ]:
#| export
def run_repl(python_code: str, shell=None) -> str:
    "Execute Python code in the REPL environment"
    shell = shell or globals().get("sh")
    if shell is None:
        return "Error: no REPL shell available"
    result = shell.run_cell(python_code)
    if result.error_in_exec or result.error_before_exec:
        return f"Error: {result.error_in_exec or result.error_before_exec}"
    output = result.stdout or ""
    if result.result is not None:
        output += str(result.result)
    return output[:5000]  # truncate long outputs

how can I check what variables have been defined here?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You can check the variables in the shell's namespace with:

```python
sh.user_ns.keys()
```

Or to see just the ones you've added (filtering out IPython's internal stuff):

```python
[k for k in sh.user_ns.keys() if not k.startswith('_')]
```

Want to try it?

In [ ]:
#| eval: false
sh.user_ns.keys()

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', 'context', '_0', 'llm_query'])

In [ ]:
#| eval: false
run_repl("llm_query(f'what is the title of the blog post? {context[:200]}')")

'The title of the blog post is **“The Not-so Bitter Lesson.”**'

This also works with the context provided and with the llm query function. Noice

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Perfect! Now you can pass `run_repl` as a tool to your outer Chat instance along with the system prompt you showed earlier.

Something like:

```python
outer_chat = Chat(model="...", tools=[run_repl])
```

Then add your system prompt and query, and the root LM should be able to interact with the REPL, inspect the context, and call `llm_query()` recursively.

Want to wire that up and test the full loop?

## The system prompt

Please give me a shorter version of the big system prompt above that is fitting for our exact implementation. I think the most important piece of information that should come across is that the LLM has access to a REPL that enables it to call another LM and that has the context preloaded as a string variable and that it should always always consult the context first before giving an answer and that it can play with the context as python. Use whatever you can from the system prompt above.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here's a condensed version:

```python
REPL_SYSTEM_PROMPT = """You are answering a query with associated context that is available in your REPL environment.

Your REPL has:
- A `context` variable (string) containing important information about your query
- An `llm_query(query: str)` function to recursively call another LLM

IMPORTANT: Always inspect the `context` variable first before answering. Use Python to explore, chunk, search, or transform it as needed.

Think step-by-step and use the REPL actively to build your answer."""
```

Does this capture what you need?

In [ ]:
#| eval: false
REPL_SYSTEM_PROMPT = """You are answering a query with associated context that is available in your REPL environment.

Your REPL has:
- A `context` variable (string) containing important information about your query
- An `llm_query(query: str)` function to recursively call another LLM

IMPORTANT: Always inspect the `context` variable first before answering. Use Python to explore, chunk, search, or transform it as needed.

Think step-by-step and use the REPL actively to build your answer."""

Let's wrap the preperation of the shell in a function

In [ ]:
#| export
def prep_shell(context, model="openai/gpt-oss-20b", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY")):
    def llm_query(query: str, max_tokens: int = 8096):
        msg = mk_msg(query)
        response = litellm.completion(model, [msg], base_url=base_url, api_key=api_key, max_tokens=max_tokens)
        return response.choices[0].message.content

    sh = get_shell()
    sh.user_ns['context'] = context
    sh.user_ns['llm_query'] = llm_query
    return sh


this is a good nested function. But, will we only get the output from the llm. this is what I am getting from the first pass. Here are the function definitions:
def prep_shell(context, model="openai/gpt-oss-20b", base_url="https://openrouter.ai/api/v1", api_key=os.getenv("OPENROUTER_API_KEY")):
    def llm_query(query: str):
        msg = mk_msg(query)
        response = litellm.completion(model, [msg], base_url=base_url, api_key=api_key)
        return response.choices[0].message.content

    sh = get_shell()
    sh.user_ns['context'] = context
    sh.user_ns['llm_query'] = llm_query
    return sh

outer_chat = Chat(
    model="openai/gpt-oss-120b",
    sp=REPL_SYSTEM_PROMPT,
    tools=[run_repl],
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)



##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good catch! Yes, with your current setup, the outer LLM will only see the text output from `llm_query()`, not the full `ModelResponse` object with usage stats, etc.

This is probably fine for the RLM use case—the root LM just needs the answer text. But if you want to track token usage or costs across all the recursive calls, you'd need to:

1. Store the full response objects somewhere (e.g., a list)
2. Return just the content to the REPL
3. Aggregate usage stats afterward

Do you need to track usage/costs, or is just getting the text output sufficient for now?

In [ ]:
#| eval: false
?Chat

```python
def Chat(
    model:str, # LiteLLM compatible model name
    sp:str='', # System prompt
    temp:int=0, # Temperature
    search:bool=False, # Search (l,m,h), if model supports it
    tools:list=None, # Add tools
    hist:list=None, # Chat history
    ns:Optional=None, # Custom namespace for tool calling
    cache:bool=False, # Anthropic prompt caching
    cache_idxs:list=[-1], # Anthropic cache breakpoint idxs, use `0` for sys prompt if provided
    ttl:NoneType=None, # Anthropic prompt caching ttl
    api_base:NoneType=None, # API base URL for custom providers
    api_key:NoneType=None, # API key for custom providers
    extra_headers:NoneType=None, # Extra HTTP headers for custom providers
    tc_refs:bool=False, # Enable tool call result references
    tc_res_eval:bool=False, # literal_eval tool results before storing in tc_res
    markup:int=0, # Cost markup multiplier (e.g. 0.5 for 50%)
):

```



```
<no docstring>
```



**File:** `/usr/local/lib/python3.12/site-packages/lisette/core.py`

**Type:** type

In [ ]:
#| eval: false
outer_chat = Chat(
    model="openai/gpt-oss-120b",
    sp=REPL_SYSTEM_PROMPT,
    tools=[run_repl],
    api_base="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

res1 = outer_chat("What made the bitter lesson click for the author?", max_tokens=8096)
res2 = outer_chat("What made the bitter lesson click for the author?")

TypeError: int() argument must be a string, a bytes-like object or a real number, not 'NoneType'

I am not sure how I can fix this? SHould I just add max_tokens to the chat argument? I changed it but I still got this error? Both routines give the same error.


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The error is in lisette's `_call` method — it does `int(max_tokens)` but `max_tokens` is `None`. Looking at the traceback, the recursive `_call` on line 482 doesn't pass `max_tokens` through, so on the second iteration it's `None`.

Try checking what `_prep_call` does:

```python
??Chat._prep_call
```

That should show us how `max_tokens` is supposed to get a default value, and why it's failing.

In [ ]:
#| eval: false
??Chat._prep_callb

Object `Chat._prep_callb` not found.


Okay, I can see that it is a conditional which is part of an exception. I had set the initial global variable to 8096. Maybe I can exclude that?


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The issue is that `model_info.get('max_output_tokens')` returns `None` for your OpenRouter model, so the fallback doesn't work. And on the tool loop's second iteration, `max_tokens` isn't passed through (look at line 482 — it's not in the args).

The cleanest workaround is to register your model with litellm so the fallback works:

```python
from litellm import register_model
register_model({"openai/gpt-oss-120b": {"max_output_tokens": 8096}})
```

That way, every time `_prep_call` sees `max_tokens=None`, it'll pick up `8096` from the model info. Want to try that?

In [ ]:
#| eval: false
from litellm import register_model
register_model({"openai/gpt-oss-120b": {"max_output_tokens": 8096}})

{'openai/gpt-oss-120b': {'max_output_tokens': 8096}}

In [ ]:
#| eval: false
outer_chat("What made the bitter lesson click for the author?")

The author’s “aha” moment came when he started playing with **tinygrad**.  
Reading George Hotz’s blog post about the library, he saw that the whole job of tinygrad—​from exposing the low‑level hardware primitives to writing fast GPU kernels and scheduling operations—was **explicitly framed as a massive search problem**.  

Tinygrad’s design treats every optimization (kernel selection, memory layout, operator fusion, etc.) as something that can be discovered by **search + raw compute** rather than by hand‑crafted, domain‑specific tricks.  

Seeing that concrete, code‑level example made the abstract claim in Sutton’s “Bitter Lesson” click for him: the lesson isn’t just a philosophical warning that clever ideas get eclipsed; it’s a practical observation that **when you expose a problem so that a search algorithm can explore it, sheer compute will eventually out‑perform hand‑engineered solutions**.  

In short, the author realized that the “search‑and‑compute” principle that Sutton described is exactly what tinygrad is built around, and that realization made the Bitter Lesson crystal‑clear.

<details>

- id: `gen-1774860352-U4oHjqKvvQNJl9PX9dff`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=313, prompt_tokens=1166, total_tokens=1479, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=90, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=0.000104944, is_byok=False, cost_details={'upstream_inference_cost': 0.000104944, 'upstream_inference_prompt_cost': 4.5474e-05, 'upstream_inference_completions_cost': 5.947e-05})`

</details>

How can we repackage the function to have all the steps in and I didn't understand why the snippet was necessary to make it work?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

**Why `register_model` was needed:** When lisette runs the tool loop, it calls itself recursively (after processing tool results). On that second call, `max_tokens` isn't passed through, so it's `None`. Then `_prep_call` tries to look up a default from litellm's model registry — but since `openai/gpt-oss-120b` isn't a standard model, it gets `None` back, and `int(None)` crashes. Registering the model gives it a fallback value to find.

**For repackaging:** You'd want a single function that takes a context and query, sets up everything (shell, llm_query, run_repl, outer chat), and returns the answer. Want me to sketch that out, or would you prefer to write it yourself?

Can you help me out to write the function? I appreciate docstrings in numpy format as well. :)

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Here you go:

```python
def rlm(query, context, 
        root_model="openai/gpt-oss-120b",
        sub_model="openai/gpt-oss-20b",
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        max_tokens=8096):
    """
    Run a Recursive Language Model query over a context.

    Uses a Python REPL environment where the context is stored as a
    variable, and the root LM can recursively call a sub-LM to
    analyze chunks of that context.

    Parameters
    ----------
    query : str
        The question or instruction to answer using the context.
    context : str
        The context string to make available in the REPL.
    root_model : str, optional
        Model for the outer/root LM that orchestrates the REPL.
    sub_model : str, optional
        Model for recursive sub-LM calls from within the REPL.
    base_url : str, optional
        API base URL for the LLM provider.
    api_key : str, optional
        API key for the LLM provider.
    max_tokens : int, optional
        Maximum output tokens for the root model.

    Returns
    -------
    str
        The root LM's final answer to the query.
    """
    register_model({root_model: {"max_output_tokens": max_tokens}})

    def llm_query(q: str):
        msg = mk_msg(q)
        resp = litellm.completion(sub_model, [msg], base_url=base_url, api_key=api_key)
        return resp.choices[0].message.content

    sh = get_shell()
    sh.user_ns['context'] = context
    sh.user_ns['llm_query'] = llm_query

    def run_repl(python_code: str) -> str:
        "Execute Python code in the REPL environment"
        result = sh.run_cell(python_code)
        if result.error_in_exec or result.error_before_exec:
            return f"Error: {result.error_in_exec or result.error_before_exec}"
        output = result.stdout or ""
        if result.result is not None:
            output += str(result.result)
        return output[:5000]

    chat = Chat(
        model=root_model, sp=REPL_SYSTEM_PROMPT, tools=[run_repl],
        api_base=base_url, api_key=api_key
    )
    return chat(query)
```

Give it a spin with something like:

```python
rlm("What made the bitter lesson click for the author?", bp)
```

In [ ]:
#| export
def rlm(query, context, 
        root_model="openai/gpt-oss-120b",
        sub_model="openai/gpt-oss-20b",
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        max_tokens=8096):
    """
    Run a Recursive Language Model query over a context.

    Uses a Python REPL environment where the context is stored as a
    variable, and the root LM can recursively call a sub-LM to
    analyze chunks of that context.

    Parameters
    ----------
    query : str
        The question or instruction to answer using the context.
    context : str
        The context string to make available in the REPL.
    root_model : str, optional
        Model for the outer/root LM that orchestrates the REPL.
    sub_model : str, optional
        Model for recursive sub-LM calls from within the REPL.
    base_url : str, optional
        API base URL for the LLM provider.
    api_key : str, optional
        API key for the LLM provider.
    max_tokens : int, optional
        Maximum output tokens for the root model.

    Returns
    -------
    str
        The root LM's final answer to the query.

    Examples
    -------
    
    """
    register_model({root_model: {"max_output_tokens": max_tokens}})

    def llm_query(q: str):
        msg = mk_msg(q)
        resp = litellm.completion(sub_model, [msg], base_url=base_url, api_key=api_key)
        return resp.choices[0].message.content

    sh = get_shell()
    sh.user_ns['context'] = context
    sh.user_ns['llm_query'] = llm_query

    def run_repl(python_code: str) -> str:
        "Execute Python code in the REPL environment"
        result = sh.run_cell(python_code)
        if result.error_in_exec or result.error_before_exec:
            return f"Error: {result.error_in_exec or result.error_before_exec}"
        output = result.stdout or ""
        if result.result is not None:
            output += str(result.result)
        return output[:5000]

    chat = Chat(
        model=root_model, sp=REPL_SYSTEM_PROMPT, tools=[run_repl],
        api_base=base_url, api_key=api_key
    )
    return chat(query)




The author’s “aha” moment came when he read George Hotz’s blog post about **tinygrad**.  

In that post Hotz frames the whole tensor‑library stack as **a massive search problem** – from finding better scheduling strategies to writing faster GPU kernels. He explains that the library’s job is to *expose* that search space and then “apply the state of the art in search. Burn compute.”  

Seeing this concrete example made the abstract claim in Richard Sutton’s “Bitter Lesson” click for the author:

* **Sutton’s claim:** General methods that rely on brute‑force search and raw compute eventually out‑perform hand‑crafted, domain‑specific solutions.  
* **Tinygrad’s illustration:** The performance gains come not from clever hand‑tuned tricks alone, but from exposing the right parameters and letting massive compute explore them automatically.  

The key insight was that **engineers still have a crucial role** – they must formulate the problem, expose the search space, and build the infrastructure that lets compute do the heavy lifting. Once the author saw tinygrad doing exactly that, Sutton’s point became clear.

<details>

- id: `gen-1774860369-phRGq59cQ2GnLvunmTOl`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=403, prompt_tokens=1132, total_tokens=1535, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=196, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=0.000120718, is_byok=False, cost_details={'upstream_inference_cost': 0.000120718, 'upstream_inference_prompt_cost': 4.4148e-05, 'upstream_inference_completions_cost': 7.657e-05})`

</details>

In [ ]:
#| eval: false
outer_chat("What made the bitter lesson click for the author?", max_steps=2)

It clicked for him when he started using **tinygrad**.  
Reading George Hotz’s blog post about the library, he saw that tinygrad’s entire purpose is to **expose the low‑level optimization work as a search problem**—searching for better schedules, faster kernels, more efficient memory layouts, etc.—and then let raw compute do the heavy lifting.  

Seeing a concrete, code‑level system that frames engineering work as “search + compute” made Sutton’s abstract claim crystal clear: once you expose the right problem space, sheer compute will out‑perform hand‑crafted, domain‑specific tricks. That realization was the author’s “aha” moment and the point at which the Bitter Lesson truly clicked.

<details>

- id: `gen-1774860377-dLvn3XbNiBP3hDGklqXJ`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=224, prompt_tokens=1417, total_tokens=1641, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=81, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=9.7823e-05, is_byok=False, cost_details={'upstream_inference_cost': 9.7823e-05, 'upstream_inference_prompt_cost': 5.5263e-05, 'upstream_inference_completions_cost': 4.256e-05})`

</details>

In [ ]:
#| eval: false
outer_chat.hist

[{'role': 'user',
  'content': 'What made the bitter lesson click for the author?'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{\n  "python_code": "print(context[:2000])"\n}', name='run_repl'), id='chatcmpl-tool-fbcd9339a72d451299dbafe3d502c169', type='function')], function_call=None, reasoning_content='We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', provider_specific_fields={'refusal': None, 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'format': 'unknown', 'index': 0}], 'reasoning': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'reasoning_content': 'We need to answer: "What made the bitter lesson click for the author?" Need context.

In [ ]:
#| eval: false
REPL_SYSTEM_PROMPT = """You are tasked with answering a query with associated context. You can access, transform, and analyze this context interactively in a REPL environment that can recursively query sub-LLMs, which you are strongly encouraged to use as much as possible. You will be queried iteratively until you provide a final answer.

Your REPL has:
- A `context` variable (string) containing important information about your query
- An `llm_query(query: str)` function to recursively call another LLM

IMPORTANT:
- Always inspect the `context` variable first before answering
- The context can be VERY LONG (millions of tokens). Never print it directly!
- Use Python to explore it safely: check length with len(), peek at slices, search with regex, chunk it
- Use `llm_query()` on manageable chunks to analyze semantics

Think step-by-step and use the REPL actively to build your answer."""

In [ ]:
#| eval: false
outer_chat("What made the bitter lesson click for the author?", max_steps=10)

It clicked for him when he started exploring **tinygrad**.  
Reading George Hotz’s blog post about the library, he realized that tinygrad treats every performance improvement—kernel selection, scheduling, memory layout, etc.—as a **search problem** that can be solved by throwing more compute at it. Seeing this concrete, code‑level example of “search + compute” made Sutton’s abstract “Bitter Lesson” crystal clear: once the problem is framed so that raw compute can explore it, brute‑force search will outpace hand‑crafted, domain‑specific tricks. That realization was the author’s “aha” moment.

<details>

- id: `gen-1774860389-H5eSe8xMVMoVYs8WZ3El`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=175, prompt_tokens=1585, total_tokens=1760, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=42, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=9.5065e-05, is_byok=False, cost_details={'upstream_inference_cost': 9.5065e-05, 'upstream_inference_prompt_cost': 6.1815e-05, 'upstream_inference_completions_cost': 3.325e-05})`

</details>

In [ ]:
#| eval: false
outer_chat.hist

[{'role': 'user',
  'content': 'What made the bitter lesson click for the author?'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{\n  "python_code": "print(context[:2000])"\n}', name='run_repl'), id='chatcmpl-tool-fbcd9339a72d451299dbafe3d502c169', type='function')], function_call=None, reasoning_content='We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', provider_specific_fields={'refusal': None, 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'format': 'unknown', 'index': 0}], 'reasoning': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'reasoning_content': 'We need to answer: "What made the bitter lesson click for the author?" Need context.

## Testing with longer context

This works somewhat but it doesn't use the LLM to recursively call a LLM. I will test this with a much longer context to see how it does

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good observation! The model is just reading through the context in chunks instead of using `llm_query()` to recursively analyze sections.

For a longer context, you'll likely see it either:
1. Hit the max_steps limit before finishing
2. Start using `llm_query()` when it realizes it can't handle everything by peeking

Testing with a much longer context (like concatenating multiple documents) should force it to use the recursive strategy. Want to try that and see if the behavior changes?

In [ ]:
#| eval: false
with open("splitter/stroke_cut.md") as f:
        stroke = f.read()


FileNotFoundError: [Errno 2] No such file or directory: 'splitter/stroke_cut.md'

In [ ]:
#| eval: false
stroke[:200]

NameError: name 'stroke' is not defined

In [ ]:
#| eval: false
outer_chat("Welche Patienten haben die Beste Evidenz für eine mechanische Thrombektomie?", max_steps=10)

**Kurzfassung**  
Die höchste Evidenz für eine mechanische Thrombektomie (MT) gibt es bei Patienten mit einem akuten ischämischen Schlaganfall, die:

| Merkmal | Detail | Evidenz‑Quelle |
|---------|--------|----------------|
| **Lokalisation** | Akute Verschluss‑**großer Gefäße** (Large‑Vessel‑Occlusion, LVO) im **vorderen Zerebralkreislauf** – ICA‑ oder M1‑Segment der MCA (gelegentlich M2, A1/A2) | MR CLEAN, ESCAPE, EXTEND‑IA, SWIFT‑PRIME, REVASCAT, THRACE |
| **Zeitfenster** | **≤ 6 h** nach Symptombeginn (Standard‑Trials) <br> **6–24 h** bei günstiger Bildgebung (kleiner Kern, gutes Kollateral‑Profil) | DAWN (6‑24 h, klinisch‑imaging‑basiert), DEFUSE‑3 (6‑16 h, perfusions‑basiert) |
| **Neurologischer Defizit‑Score** | **NIHSS ≥ 6** (oft ≥ 10) – deutliche neurologische Ausfälle | Alle großen Randomisiert‑Kontroll‑Studien |
| **Bildgebung** | <ul><li>CT‑Angiographie (oder MR‑Angiographie) mit Nachweis eines LVO</li><li>CT‑Perfusion / MR‑Perfusion oder DWI‑/FLAIR‑Mismatch, der einen **kleinen ischämischen Kern** (ASPECTS ≥ 6, Kern‑Volumen ≤ 70 ml) und **ausreichende penumbra** (Perfusions‑Defizit‑zu‑Kern‑Verhältnis ≥ 1,5) zeigt</li></ul> | DAWN, DEFUSE‑3, EXTEND‑IA, ESCAPE |
| **Alter** | Keine obere Altersgrenze; Studien schlossen Patienten bis 90 Jahre ein | Alle Trials |
| **Vorherige Therapie** | Kann gleichzeitig **IV‑tPA** erhalten (wenn innerhalb des 4,5‑h‑Fensters) – additive Wirkung | SWIFT‑PRIME, REVASCAT, EXTEND‑IA |
| **Ausschlusskriterien** | Massive Infarkt‑Kern‑Volumen, schwere Blutungsneigung, unkontrollierter Blutdruck, schwere Kontraindikationen für Kontrastmittel, sehr schlechte Kollateralen, terminale Erkrankungen mit kurzer Lebenserwartung | Studien‑Protokolle |

---

### Warum gerade diese Patientengruppe?

1. **Große Gefäßverschlüsse** sind die einzigen, bei denen ein mechanisches Entfernen des Thrombus (Stent‑Retriever oder Aspiration) signifikant das funktionelle Ergebnis verbessert (mOR ≈ 2,5‑3,0 für gutes Outcome, mRS 0‑2).  
2. **Zeitkritisch**: Der Nutzen nimmt mit jeder Minute ab (≈ 1,9 % schlechteres Outcome pro Minute Verzögerung). Deshalb ist das 6‑Stunden‑Fenster das „klassische“ Zeitfenster, das in den ersten sieben großen RCTs definiert wurde.  
3. **Bildbasierte Selektion** (DAWN, DEFUSE‑3) hat gezeigt, dass Patienten mit **kleinem Kern‑/großer Penumbra‑Verhältnis** auch bis zu 24 h nach Symptombeginn noch stark von einer MT profitieren.  
4. **Starke klinische Defizite** (NIHSS ≥ 6) korrelieren mit einem LVO und garantieren, dass ein potenzieller Nutzen der Reperfusion klinisch relevant ist.  
5. **Kombination mit IV‑tPA** (wenn zulässig) erhöht die Rezidiv‑Reperfusionsrate, ohne das Sicherheitsprofil zu verschlechtern.

---

### Praktische Umsetzung im klinischen Alltag  

| Schritt | Was zu tun ist |
|--------|----------------|
| **1. Schnell‑Screening** | NIHSS ≥ 6, Beginn < 6 h (oder < 24 h bei möglicher erweiterten Auswahl) |
| **2. Bildgebung** | <ul><li>CT‑Head + CT‑Angio (oder MR‑Angio) → LVO nachweisen</li><li>CT‑Perfusion (oder MR‑Perfusion/DWI‑FLAIR‑Mismatch) → Kern‑/Penumbra‑Analyse, ASPECTS ≥ 6</li></ul> |
| **3. Therapie‑Entscheidung** | Wenn LVO + günstige Bildgebung → MT (so schnell wie möglich). Bei < 4,5 h zusätzlich IV‑tPA, sofern keine Kontraindikationen. |
| **4. Nachsorge** | Kontrolle auf Blutungen, neurologische Reevaluation, ggf. Intensiv‑ oder Stroke‑Unit‑Betreuung, Rehabilitation. |

---

### Schlüssel‑Studien (Kurz‑Übersicht)

| Studie | Patienten | Zeitfenster | Ergebnis (mRS 0‑2) |
|--------|-----------|-------------|--------------------|
| **MR CLEAN** (2015) | 500 | ≤ 6 h | 33 % vs 19 % (Kontrolle) |
| **ESCAPE** (2015) | 316 | ≤ 6 h | 53 % vs 29 % |
| **EXTEND‑IA** (2015) | 35 | ≤ 6 h | 71 % vs 40 % |
| **SWIFT‑PRIME** (2015) | 190 | ≤ 6 h | 60 % vs 35 % |
| **REVASCAT** (2015) | 206 | ≤ 8 h | 43 % vs 27 % |
| **THRACE** (2016) | 414 | ≤ 6 h | 46 % vs 30 % |
| **DAWN** (2018) | 206 | 6‑24 h, klinisch‑imaging‑basiert | 49 % vs 13 % |
| **DEFUSE‑3** (2018) | 182 | 6‑16 h, perfusions‑basiert | 45 % vs 17 % |

---

## Fazit

**Die beste Evidenz für eine mechanische Thrombektomie besteht bei Patienten mit einem akuten großen Gefäßverschluss im vorderen Zerebralkreislauf, einem moderaten bis schweren neurologischen Defizit (NIHSS ≥ 6), einem kleinen ischämischen Kern (ASPECTS ≥ 6, Kern‑Volumen ≤ 70 ml) und guten Kollateralen – und das sowohl im klassischen 0‑6‑Stunden‑Fenster als auch, bei entsprechender Bildgebung, bis zu 24 Stunden nach Symptombeginn.**  

Diese Kriterien leiten die aktuelle Leitlinie (z. B. ESO‑2023, AHA/ASA‑2024) und sollten in jedem Stroke‑Team als Basis für die Entscheidung zur sofortigen MT dienen.

<details>

- id: `gen-1773216359-MNJbh9iqXmGXnPegz7KP`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=1738, prompt_tokens=986, total_tokens=2724, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=179, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, cache_write_tokens=0, video_tokens=0), cost=0.00071442, is_byok=False, cost_details={'upstream_inference_cost': 0.00071442, 'upstream_inference_prompt_cost': 8.874e-05, 'upstream_inference_completions_cost': 0.00062568})`

</details>

My output doesn't look like this with the same function


##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I notice you're getting a different output format than expected. Let me help you troubleshoot this.

**Quick questions:**

1. **What does your output look like?** (Can you show me what you're seeing?)

2. **Are you using the exact same `outer_chat()` call?**
   ```python
   outer_chat("Welche Patienten haben die Beste Evidenz für eine mechanische Thrombektomie?", max_steps=10)
   ```

3. **Is your `stroke` variable loaded with the same content?** (Check `len(stroke)` to see if it matches)

The output I showed has a nicely formatted table and structured response. If yours looks different, it could be:
- Different LLM model/settings
- Different context being passed
- Different system prompt in your `outer_chat` function

Show me what you're getting and we can figure out the difference! 🔍

In [ ]:
#| eval: false
outer_chat.hist

[{'role': 'user',
  'content': 'What made the bitter lesson click for the author?'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"python_code": "print(context[:1000])"}', name='run_repl'), id='call_ea01fd66f7f54837bc773c13', type='function')], function_call=None, reasoning_content='We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', provider_specific_fields={'refusal': None, 'reasoning': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context variable.', 'format': 'unknown', 'index': 0}], 'reasoning_content': 'We need to answer: "What made the bitter lesson click for the author?" Need context. Let\'s inspect context

## Improving the system prompt

We need to be more clear with how to use the LLM and thus we need to add an examples to the system prompt above how the models can actually use the `llm_query` tool in the REPL. Let's grab the examples from the original system prompt and improve the system prompt even more

In [ ]:
#| export
REPL_SYSTEM_PROMPT = """You are tasked with answering a query with associated context. You can access, transform, and analyze this context interactively in a REPL environment that can recursively query sub-LLMs, which you are strongly encouraged to use as much as possible. You will be queried iteratively until you provide a final answer.

Your REPL has:
- A `context` variable (string) containing important information about your query
- An `llm_query(query: str)` function that allows you to query an LLM (that can handle around 500K chars) inside your REPL environment.

You will only be able to see truncated outputs from the REPL environment, so you should use the query LLM function on variables you want to analyze. You will find this function especially useful when you have to analyze the semantics of the context. Use these variables as buffers to build up your final answer.

Make sure to explicitly look through the entire context in REPL before answering your query. An example strategy is to first look at the context and figure out a chunking strategy, then break up the context into smart chunks, and query an LLM per chunk with a particular question and save the answers to a buffer, then query an LLM with all the buffers to produce your final answer.

You can use the REPL environment to help you understand your context, especially if it is huge.

Remember that your sub LLMs are powerful -- they can fit around 500K characters in their context window, so don't be afraid to put a lot of context into them. For example, a viable strategy is to feed 10 documents per sub-LLM query. Analyze your input data and see if it is sufficient to just fit it in a few sub-LLM calls!

For example, say we want our recursive model to search for the magic number in the context (assuming the context is a string), and the context is very long, so we want to chunk it:
```python
chunk = context[:10000]
answer = llm_query(f"What is the magic number in the context? Here is the chunk: {{chunk}}")
print(answer)
```

As another example, after analyzing the context and realizing its separated by Markdown headers, we can maintain state through buffers by chunking the context by headers, and iteratively querying an LLM over it:
```python
# After finding out the context is separated by Markdown headers, we can chunk, summarize, and answer
import re
sections = re.split(r'### (.+)', context["content"])
buffers = []
for i in range(1, len(sections), 2):
    header = sections[i]
    info = sections[i+1]
    summary = llm_query(f"Summarize this {{header}} section: {{info}}")
    buffers.append(f"{{header}}: {{summary}}")
final_answer = llm_query(f"Based on these summaries, answer the original query: {{query}}\\n\\nSummaries:\\n" + "\\n".join(buffers))
```

IMPORTANT:
- Always inspect the `context` variable first before answering
- The context can be VERY LONG (millions of tokens). Never print it directly!
- Use Python to explore it safely: check length with len(), peek at slices, search with regex, chunk it
2. A `llm_query` function that allows you to query an LLM (that can handle around 500K chars) inside your REPL environment.

Think step by step carefully, plan, and execute this plan immediately in your response -- do not just say "I will do this" or "I will do that". Output to the REPL environment and especially the recursive LLM (`llm_query` in the REPL) as much as possible. Remember to explicitly answer the original query in your final answer."""

In [ ]:
#| eval: false
sh = prep_shell(stroke)

outer_chat = Chat(model="groq/openai/gpt-oss-120b", tools=[run_repl], sp=REPL_SYSTEM_PROMPT)

outer_chat("Welche Patienten haben die Beste Evidenz für eine mechanische Thrombektomie?", max_steps=10)

NameError: name 'stroke' is not defined

I guess gpt-oss-120b is just not a fan of using the llm_query function :D 

## Things to improve
* Wrap this up in a class with proper state management
* Improve the system prompt after testing it on more complicated tasks. I guess getting inspiration from the other repos that implemented this for the prompts may be valuable as well.

What do you think of wrapping this all up in a class? This makes state management easier I guess

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Good idea! A class would help manage:

1. **Shell lifecycle** - Initialize once, reuse across queries
2. **Model configuration** - Set root/recursive models in one place  
3. **Context loading** - Separate setup from query execution
4. **System prompt** - Make it configurable per instance

Something like:

```python
rlm = RecursiveLM(context=my_doc, root_model="...", recursive_model="...")
response = rlm.query("What is X?")
```

Do you want to sketch out the class structure yourself, or would you like me to show you a basic implementation?

## wrapping it up in a class

Ok, based on the context. What is missing in RLM class here? Do some edits?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let me look at the class more carefully:

<details class='tool-usage-details'>
<summary>view_msg(id='_76d53f71')</summary>

```json
{
  "id": "toolu_01VK3r5SJt4g5N17gSg6M7mu",
  "call": {
    "function": "view_msg",
    "arguments": {
      "id": "_76d53f71"
    }
  },
  "result": "<TRUNCATED>\u2026class RLM:\\n     2 \u2502     def __init__(self, context: str, root_model = \"openai/gpt-oss-120b\", recursive_model = None, temperature=0.0, base_url=\"https://openrouter.ai/api/v1\", api_key=os.getenv(\"OPENROUTER_API_KEY\")):\\n     3 \u2502         self.context = context\\n     4 \u2502         self.root_model = root_model\\n     5 \u2502         self.recursive_model = recursive_model if recursive_model else root_model\\n     6 \u2502         self.shell = self._prep_shell()\\n     7 \u2502         self.base_url = \"https://openrouter.ai/api/v1\"\\n     8 \u2502         self.api_key = os.getenv(\"OPENROUTER_API_KEY\")\\n     9 \u2502 \\n    10 \u2502         def run_repl(python_code: str) -> str:\\n    11 \u2502             \"Execute Python code in the REPL environment\"\\n    12 \u2502             result = self.shell.run_cell(python_code)\\n    13 \u2502             if result.error_in_exec or result.error_before_exec:\\n    14 \u2502                 return f\"Error: {result.error_in_exec or result.error_before_exec}\"\\n    15 \u2502             output = re\u2026</TRUNCATED>"
}
```

</details>

I see a few issues:

1. **Line 7-8**: `base_url` and `api_key` are hardcoded instead of using the parameters — and they're set *after* `_prep_shell()` runs on line 6, so `_prep_shell` can't use them
2. **Line 20**: `Chat` doesn't receive `api_base`/`api_key`
3. **Line 26**: `llm_query` doesn't pass `base_url`/`api_key` to `litellm.completion`
4. **Line 18**: No output truncation (we had `[:5000]` before)
5. Missing `register_model` call (the `int(None)` fix)

<details class='tool-usage-details'>
<summary>msg_replace_lines(id='_76d53f71', start_line=3, end_line=8, new_content='<…>…xt = context\n        self.root…</…>')</summary>

```json
{
  "id": "toolu_01N32bt3sajRNA65cUqfQDaN",
  "call": {
    "function": "msg_replace_lines",
    "arguments": {
      "id": "_76d53f71",
      "start_line": "3",
      "end_line": "8",
      "new_content": "self.context = context\n        self.root_model = root_model\n        self.recursive_model = recursive_model if recursive_model else root_model\n        self.base_url = base_url\n        self.api_key = api_key\n        register_model({root_model: {\"max_output_tokens\": 8096}})\n        self.shell = self._prep_shell()"
    }
  },
  "result": "'@@ -5,5 +5,6 @@\\n         self.recursive_model = recursive_model if recursive_model else root_model\\n+        self.base_url = base_url\\n+        self.api_key = api_key\\n+        register_model({root_model: {\"max_output_tokens\": 8096}})\\n         self.shell = self._prep_shell()\\n-        self.base_url = \"https://openrouter.ai/api/v1\"\\n-        self.api_key = os.getenv(\"OPENROUTER_API_KEY\")\\n '"
}
```

</details>



<details class='tool-usage-details'>
<summary>msg_str_replace(id='_76d53f71', old_str='<…>…put\n\n        self.chat = Chat(…</…>', new_str='<…>…put[:5000]\n\n        self.chat …</…>')</summary>

```json
{
  "id": "toolu_016xfFX6kpFAA7ctpNo9LJyi",
  "call": {
    "function": "msg_str_replace",
    "arguments": {
      "id": "_76d53f71",
      "old_str": "return output\n\n        self.chat = Chat(model=self.root_model, tools=[run_repl], sp=REPL_SYSTEM_PROMPT, temp=temperature)",
      "new_str": "return output[:5000]\n\n        self.chat = Chat(model=self.root_model, tools=[run_repl], sp=REPL_SYSTEM_PROMPT, temp=temperature,\n                         api_base=self.base_url, api_key=self.api_key)"
    }
  },
  "result": "'@@ -18,5 +18,6 @@\\n                 output += str(result.result)\\n-            return output\\n+            return output[:5000]\\n \\n-        self.chat = Chat(model=self.root_model, tools=[run_repl], sp=REPL_SYSTEM_PROMPT, temp=temperature)\\n+        self.chat = Chat(model=self.root_model, tools=[run_repl], sp=REPL_SYSTEM_PROMPT, temp=temperature,\\n+                         api_base=self.base_url, api_key=self.api_key)\\n '"
}
```

</details>



<details class='tool-usage-details'>
<summary>msg_str_replace(id='_76d53f71', old_str='<…>…ellm.completion(self.recursive…</…>', new_str='<…>…ellm.completion(self.recursive…</…>')</summary>

```json
{
  "id": "toolu_01CN7fKv9UDhUYn7vWyJwwAs",
  "call": {
    "function": "msg_str_replace",
    "arguments": {
      "id": "_76d53f71",
      "old_str": "return litellm.completion(self.recursive_model, [msg]).choices[0].message.content",
      "new_str": "return litellm.completion(self.recursive_model, [msg], base_url=self.base_url, api_key=self.api_key).choices[0].message.content"
    }
  },
  "result": "'@@ -27,3 +27,3 @@\\n             msg = mk_msg(query)\\n-            return litellm.completion(self.recursive_model, [msg]).choices[0].message.content\\n+            return litellm.completion(self.recursive_model, [msg], base_url=self.base_url, api_key=self.api_key).choices[0].message.content\\n '"
}
```

</details>

Here's a summary of the fixes:

1. **Moved `base_url`/`api_key` assignment before `_prep_shell()`** so they're available when setting up the shell
2. **Used the parameters** instead of hardcoding them
3. **Added `register_model`** to avoid the `int(None)` crash
4. **Passed `api_base`/`api_key` to `Chat`** and to `litellm.completion` in `llm_query`
5. **Added `[:5000]` truncation** back to `run_repl`

Give it a run and see how it goes!

In [ ]:
#| eval: false
rlm = RLM(context=bp, root_model="openrouter/openai/gpt-4o-mini", temperature=0.2)

In [ ]:
#| eval: false
rlm("what were the 3 take at home messages in the blogpost", max_steps=10)

I've analyzed the context of the blog post titled "The Not-so Bitter Lesson." Here are the three key take-home messages from the post:

1. **Effective Problem Formulation**: Engineers need to focus on formulating problems in a way that exposes the right parameters for algorithms to explore. This involves designing problem representations that make search tractable.

2. **Building Evaluation Mechanisms**: It's crucial to create robust evaluation systems that can measure the effectiveness of solutions. This allows the search mechanism to receive feedback and refine its approach.

3. **Defining Constraints**: Engineers should set constraints that solutions must satisfy, such as memory and time limits. This helps guide the search process and ensures that the solutions are practical and safe.

The overall message emphasizes that while raw compute and search can outperform domain-specific solutions, the role of engineers is still vital in framing problems, building evaluations, and defining constraints to enable effective search and compute. 

If you need further details or specific examples from the blog post, please let me know!

<details>

- id: `gen-1774862668-uI0GQ05xJQfaXQo8GY4K`
- model: `openai/gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=205, prompt_tokens=3122, total_tokens=3327, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=2560, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=0.0003993, is_byok=False, cost_details={'upstream_inference_cost': 0.0003993, 'upstream_inference_prompt_cost': 0.0002763, 'upstream_inference_completions_cost': 0.000123})`

</details>

In [ ]:
#| eval: false
# rlm = RLM(context=massive_context, root_model="openrouter/openai/gpt-4o-mini", temperature=0)
# rlm("what is the magic number?", max_steps=150)

I have thoroughly examined the context, which is extensive and contains a variety of information. However, I did not find a specific "magic number" mentioned in the context. The content primarily consists of random data, information about various topics, and repeated phrases, but it lacks a clear definition or identification of a "magic number."

To proceed, I recommend the following steps:
1. **Clarification**: If you have a specific definition or context for what you mean by "magic number," please provide that information.
2. **Focused Search**: If there are specific sections or keywords related to the "magic number" that you would like me to search for, please specify those.

Once I have more targeted information or direction, I can perform a more focused search within the context to find the relevant details.

<details>

- id: `gen-1774862078-3JqFj2ItlDejnU4WmVny`
- model: `openai/gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=162, prompt_tokens=30646, total_tokens=30808, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=30336, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=0.0024189, is_byok=False, cost_details={'upstream_inference_cost': 0.0024189, 'upstream_inference_prompt_cost': 0.0023217, 'upstream_inference_completions_cost': 9.72e-05})`

</details>

I want to make an example that review needle in a haystack and tool calling with the new class. How do I go about that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Great idea for testing! Here's how I'd approach each:

**Needle in a haystack:** Generate a long context (e.g. lots of random/filler text), hide a specific fact somewhere in the middle (like "The secret code is 42"), then ask the RLM to find it. This forces the model to actually search through the context rather than answering from its own knowledge.

**Tool calling:** Use a context that's too long or complex for the root LM to handle in one peek, so it *has* to use `llm_query()` to analyze chunks. For example, multiple documents where the answer requires synthesizing info across them.

Which one do you want to build first?

Needle in a haystack

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Let's build it step by step.

First, you'll need to generate a big chunk of filler text with a hidden "needle" buried somewhere in the middle. A simple approach:

1. Use a bunch of repeated paragraphs of lorem ipsum or random sentences as the "haystack"
2. Insert a unique fact (the "needle") at a random position
3. Query the RLM to find it

How long do you want the haystack to be? Something like 50K characters would be a good start to force the model to chunk and search rather than reading it all at once.

In [ ]:
#| eval: false
massive_context = generate_massive_context()
print(massive_context[:1000])

Generating massive context with 1M lines...


Magic number inserted at position 452409
text sample content random data information blah blah
sample random text sample data data text
sample blah blah
content sample text
content sample data random random information information
text blah information information text content text data
blah information content text
content text text information data content text
random content data sample random content blah
random content random random text blah sample
content information data
blah data blah content content information information
random information random blah blah random
random data sample information information
data random data
information content data
data data text random content data
content information blah information
random text random random blah text content content
content text sample content information information
sample text information random information sample sample
information content sample sample sample
blah text text information random data
sample blah data
tex

Can you make me mermaid diagram of the architecture of an RLM with lisette to Grok this?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

Sure! Here's a Mermaid diagram of your RLM architecture:

```mermaid
graph TD
    U[👤 User Query] --> RLM[🧠 RLM Class]
    
    RLM --> Chat[💬 lisette Chat<br/>Root Model]
    Chat -->|System Prompt| SP[📋 REPL System Prompt]
    
    Chat -->|Tool Call| RunREPL[🔧 run_repl tool]
    RunREPL -->|python_code| Shell[🐚 toolslm Shell<br/>IPython REPL]
    Shell -->|result/stdout| RunREPL
    RunREPL -->|truncated output| Chat
    
    Shell --- Context[📄 context variable<br/>user_ns]
    Shell --- LLMQuery[🔁 llm_query function<br/>user_ns]
    
    LLMQuery -->|litellm.completion| SubLM[🤖 Sub-Model<br/>Recursive LM]
    SubLM -->|content string| LLMQuery
    
    Chat -->|Final Answer| RLM
    RLM --> R[📝 Response]

    style RLM fill:#f9f,stroke:#333
    style Shell fill:#bbf,stroke:#333
    style Chat fill:#bfb,stroke:#333
    style SubLM fill:#fbb,stroke:#333
```

The key flow is:
- **Green**: Root LM orchestrates via lisette `Chat` with tool calling
- **Blue**: Python REPL holds `context` + `llm_query` in its namespace
- **Red**: Recursive sub-LM calls happen *inside* the REPL via `llm_query()`

Error: Error: Parse error on line 5:
...lisette Chat="" -->|content="" -->|fina
----------------------^
Expecting 'SQE', 'TAGEND', 'UNICODE_TEXT', 'TEXT', 'TAGSTART', got 'PIPE'

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The `<br/>` tags and special characters are causing parsing issues. Here's a fixed version:

```mermaid
graph TD
    U["👤 User Query"] --> RLM["RLM Class"]
    
    RLM --> Chat["💬 lisette Chat - Root Model"]
    Chat -->|System Prompt| SP["📋 REPL System Prompt"]
    
    Chat -->|Tool Call| RunREPL["🔧 run_repl tool"]
    RunREPL -->|python_code| Shell["🐚 toolslm Shell - IPython REPL"]
    Shell -->|result/stdout| RunREPL
    RunREPL -->|truncated output| Chat
    
    Shell --- Context["📄 context variable"]
    Shell --- LLMQuery["🔁 llm_query function"]
    
    LLMQuery -->|litellm.completion| SubLM["🤖 Sub-Model - Recursive LM"]
    SubLM -->|content string| LLMQuery
    
    Chat -->|Final Answer| RLM
    RLM --> R["📝 Response"]

    style RLM fill:#f9f,stroke:#333
    style Shell fill:#bbf,stroke:#333
    style Chat fill:#bfb,stroke:#333
    style SubLM fill:#fbb,stroke:#333
```

The fix: wrap all node labels in double quotes to escape special characters.

In [ ]:
#| export
from IPython.display import HTML

# Load Mermaid library once

def tool_mermaid(graph:str # string representing the chart to be displayed
    )->dict:
    '''Returns two blocks:
    - html_graph: this one need to be added where you want to dispaly the chart
    - html_run: this one need to be added at the end of the message and only "once" even if you generate multiple charts
    The result should be returned inline with the response without any markdown codeblock.
    ''' 
    return dict(html_graph=f'<div class="mermaid">{graph}</div>',
                html_run='<script>if (window.mermaid) mermaid.run();</script>') # fix thnx to @dgwyer

HTML(
<script src="https://cdn.jsdelivr.net/npm/mermaid@11/dist/mermaid.min.js"></script>
<script>mermaid.initialize({ startOnLoad: false });</script>
)

In [ ]:
#| export
def mm(graph): return HTML('\n'.join(tool_mermaid(graph).values()))

In [ ]:
#| eval: false
mm(
"""
graph TD
    U["👤 User Query"] --> RLM["RLM Class"]
    
    RLM --> Chat["💬 lisette Chat - Root Model"]
    Chat -->|System Prompt| SP["📋 REPL System Prompt"]
    
    Chat -->|Tool Call| RunREPL["🔧 run_repl tool"]
    RunREPL -->|python_code| Shell["🐚 toolslm Shell - IPython REPL"]
    Shell -->|result/stdout| RunREPL
    RunREPL -->|truncated output| Chat
    
    Shell --- Context["📄 context variable"]
    Shell --- LLMQuery["🔁 llm_query function"]
    
    LLMQuery -->|litellm.completion| SubLM["🤖 Sub-Model - Recursive LM"]
    SubLM -->|content string| LLMQuery
    
    Chat -->|Final Answer| RLM
    RLM --> R["📝 Response"]

    style RLM fill:#f9f,stroke:#333
    style Shell fill:#bbf,stroke:#333
    style Chat fill:#bfb,stroke:#333
    style SubLM fill:#fbb,stroke:#333
"""
)

HTML(<div class="mermaid">
graph TD
    U["👤 User Query"] --> RLM["RLM Class"]

    RLM --> Chat["💬 lisette Chat - Root Model"]
    Chat -->|System Prompt| SP["📋 REPL System Prompt"]

    Chat -->|Tool Call| RunREPL["🔧 run_repl tool"]
    RunREPL -->|python_code| Shell["🐚 toolslm Shell - IPython REPL"]
    Shell -->|result/stdout| RunREPL
    RunREPL -->|truncated output| Chat

    Shell --- Context["📄 context variable"]
    Shell --- LLMQuery["🔁 llm_query function"]

    LLMQuery -->|litellm.completion| SubLM["🤖 Sub-Model - Recursive LM"]
    SubLM -->|content string| LLMQuery

    Chat -->|Final Answer| RLM
    RLM --> R["📝 Response"]

    style RLM fill:#f9f,stroke:#333
    style Shell fill:#bbf,stroke:#333
    style Chat fill:#bfb,stroke:#333
    style SubLM fill:#fbb,stroke:#333
</div>
<script>if (window.mermaid) mermaid.run();</script>)

How's the function now?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I can see a few issues in your `RLM` class:

1. **`_prep_shell()` doesn't return anything** - It defines `llm_query` but never creates or returns a shell
2. **The shell needs the context and `llm_query` injected** - Like you did earlier with `prep_shell()`

Here's what `_prep_shell()` should look like:

```python
def _prep_shell(self):
    shell = InteractiveShell()
    
    def llm_query(query: str):
        msg = mk_msg(query)
        response = litellm.completion(
            model=self.recursive_model,
            messages=[msg],
            base_url=self.base_url,
            api_key=self.api_key
        )
        return response.choices[0].message.content
    
    shell.push({'context': self.context, 'llm_query': llm_query})
    return shell
```

Also, you're setting `self.base_url` and `self.api_key` as instance variables but they're already passed as parameters - you can just use `self.base_url = base_url` etc.

Want to fix these and try it out?

How would the Class look like? I have also notice an interactiveshell has been added

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Here's what your complete `RLM` class should look like with the fixes:

```python
class RLM:
    def __init__(self, context: str, root_model="openai/gpt-oss-120b", 
                 recursive_model=None, temperature=0.0, 
                 base_url="https://openrouter.ai/api/v1", 
                 api_key=None):
        self.context = context
        self.root_model = root_model
        self.recursive_model = recursive_model if recursive_model else root_model
        self.base_url = base_url
        self.api_key = api_key or os.getenv("OPENROUTER_API_KEY")
        self.shell = self._prep_shell()
        
        def run_repl(python_code: str) -> str:
            "Execute Python code in the REPL environment"
            result = self.shell.run_cell(python_code)
            if result.error_in_exec or result.error_before_exec:
                return f"Error: {result.error_in_exec or result.error_before_exec}"
            output = result.stdout or ""
            if result.result is not None:
                output += str(result.result)
            return output
        
        self.chat = Chat(model=self.root_model, tools=[run_repl], 
                        sp=REPL_SYSTEM_PROMPT, temp=temperature)
    
    def _prep_shell(self):
        shell = InteractiveShell()
        
        def llm_query(query: str):
            msg = mk_msg(query)
            response = litellm.completion(
                model=self.recursive_model,
                messages=[msg],
                base_url=self.base_url,
                api_key=self.api_key
            )
            return response.choices[0].message.content
        
        shell.push({'context': self.context, 'llm_query': llm_query})
        return shell
    
    def __call__(self, query: str, max_steps=10):
        return self.chat(query, max_steps=max_steps)
```

The key fix is that `_prep_shell()` now properly returns the shell after setting it up.

**Note:** Some earlier messages were removed due to context size limits - you originally had 82 messages, now showing 19.

Want to test this class with your stroke document?

In [ ]:
#| eval: false
print(bp[:100])

---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tiny


How do I fix this error?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The error says the API key for OpenAI isn't set. The issue is that you're using `openai/gpt-5-nano` as your model, which routes through OpenAI's API (not OpenRouter), but you haven't set the `OPENAI_API_KEY` environment variable.

You have two options:

**Option 1:** Use an OpenRouter model instead (since you have that key set):
```python
rlm = RLM(context=bp, root_model="openai/gpt-4o-mini", temperature=1)
```

**Option 2:** Set your OpenAI API key:
```python
import os
os.environ["OPENAI_API_KEY"] = "your-openai-key-here"
```

Which approach do you prefer?

---

**Note:** Some earlier messages were removed due to context size limits (originally 88 messages, now 25 showing).

## Test with even longer context.

Let's add the carotis guideline to our context to make it even more challenging to find stuff.

In [ ]:
#| eval: false
import json

In [ ]:
#| eval: false
with open("splitter/carotis_cut.md", "r") as f:
    carotis = f.read()

carotis[:200]

FileNotFoundError: [Errno 2] No such file or directory: 'splitter/carotis_cut.md'

In [ ]:
#| eval: false
len(carotis) + len(stroke)

NameError: name 'carotis' is not defined

Yeah okay so that is some hefty context :D Let's see how many tokens this would be for GPT-4 using tiktoken

In [ ]:
#| eval: false
import tiktoken

In [ ]:
#| eval: false
enc = tiktoken.get_encoding("cl100k_base")  # GPT-4/GPT-3.5 tokenizer
combined = stroke + carotis
num_tokens = len(enc.encode(combined))
print(f"Total tokens: {num_tokens:,}")

NameError: name 'stroke' is not defined

Great so the context window of gpt oss 120b is around 130k tokens, which means we are way above the usual context window.

In [ ]:
#| eval: false
q = """Welche Empfehlungen gibt es zur frühen dualen Plättchenhemmung bei Patienten mit leichtem nicht-kardioembolischem ischämischem Schlaganfall oder TIA mit hohem Rezidivrisiko?
"""

rlm = RLM(context="\n\n".join([carotis, stroke]))
rlm(q)

NameError: name 'carotis' is not defined

Still works. Great. 

Now let's add a way that makes reading the trajectories or the history a bit more pleasant. Let's create a function that converts the trajectory in nice markdown

## Pretty printing `lisette` trajectories

I want to better understand what the model is doing in it's trace so the next step will be to "pretty print" the trajectory, maybe even in a web UI.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Good idea! Visualizing the trajectory will help you understand the model's reasoning flow. A few approaches:

**Simple text-based:**
- Print each step with indentation (REPL calls, `llm_query()` calls, results)
- Show truncated outputs to keep it readable

**Web UI:**
- Display as an expandable tree (root → REPL → recursive LLM calls)
- Color-code by type (search, extract, recursive query, final answer)
- Show token counts and timing per step

**What to show:**
- Each REPL code execution + output
- Each `llm_query()` call with the prompt and response
- The final answer

Do you want to start with a simple text pretty-printer first, or jump straight to building a web UI? I can help with either approach.

In [ ]:
#| eval: false
rlm.chat.hist

[{'role': 'user',
  'content': 'what were the 3 take at home messages in the blogpost'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"python_code":"len(context)"}', name='run_repl'), id='call_UN2wVHsyd5uq4w2ggdi4T8wt', type='function')], function_call=None, provider_specific_fields={'reasoning': None, 'refusal': None}),
 {'tool_call_id': 'call_UN2wVHsyd5uq4w2ggdi4T8wt',
  'role': 'tool',
  'name': 'run_repl',
  'content': '9245'},
 Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"python_code":"context[:1000]"}', name='run_repl'), id='call_7AIO3CWtPsioWZvjzx5nBJ8l', type='function')], function_call=None, provider_specific_fields={'reasoning': None, 'refusal': None}),
 {'tool_call_id': 'call_7AIO3CWtPsioWZvjzx5nBJ8l',
  'role': 'tool',
  'name': 'run_repl',
  'content': '---\ntitle: "The Not-so Bitter Lesson"\ndate: 2025-10-20T15:27:52+01:

In [ ]:
#| eval: false
x=rlm.chat.hist[1]
x

Message(content=None, role='assistant', tool_calls=[ChatCompletionMessageToolCall(index=0, function=Function(arguments='{"python_code":"len(context)"}', name='run_repl'), id='call_UN2wVHsyd5uq4w2ggdi4T8wt', type='function')], function_call=None, provider_specific_fields={'reasoning': None, 'refusal': None})

In [ ]:
#| eval: false
isinstance(x, litellm.types.utils.Message)

True

In [ ]:
#| eval: false
md = ""

md += f"### {x['role']}\n\n"
if x["content"]:
    md += (x["content"] + "\n\n")

if "tool_calls" in x:
    c = json.loads(x["tool_calls"][0]["function"]["arguments"])["python_code"]
    md += f"```\n{c}\n```\n\n"

md+= "---"

print(md)

### assistant

---


In [ ]:
#| export
def print_history(rlm):
    return Markdown("\n".join(map(_md, rlm.chat.hist)))

In [ ]:
#| eval: false
_md(rlm.chat.hist[1])

'### assistant\n\n```\nlen(context)\n```\n\n---'

In [ ]:
#| export
from IPython.display import Markdown

In [ ]:
#| eval: false
Markdown("\n".join(map(_md, rlm.chat.hist)))

<div class="prose">

### user

what were the 3 take at home messages in the blogpost

---
### assistant

```
len(context)
```

---
### tool

9245

---
### assistant

```
context[:1000]
```

---
### tool

---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tinygrad helped me finally understand what Sutton meant
---

Have you heard about The Bitter Lesson? If you are on Twitter and follow some ML/AI people, I bet you have.

If you haven't: It is a [blog post by Richard Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html), a Canadian computer scientist, who wrote one of *the* books about reinforcement learning.

Sutton's core argument is simple: general methods that leverage search and compute outperform domain-specific solutions. By "search," he doesn't mean information retrieval - he means using learning algorithms to explore possible solutions or strategies. His go-to example is computer chess: early engines relied heavily on chess expertise to evaluate positions, while AlphaZero knew only the basic rules and learned by playing itself, eventually surpassing any human player.

This lesson is "bitter" because it suggests our cleve

---
### assistant

```
context[1000:2000]
```

---
### tool

r insights don't matter. Our domain-specific knowledge gets crushed by what looks like "dumb brute-force search and compute." If you're an engineer, this naturally raises an uncomfortable question: what's the point of us?

## The Click

For a long time, I didn't really understand Sutton's point. What does it mean to solve problems using search and compute? What is the role of engineers then? Are we just supposed to prompt a LLM and hope for the best? That can't scale.

My aha moment came from an unlikely source: [`tinygrad`](https://github.com/tinygrad/tinygrad). It's a Python library similar to PyTorch but much simpler on the surface. I think George Hotz initially built it as a toy project after watching Andrej Karpathy create micrograd, but it evolved into a solid tensor library that runs LLMs efficiently on different hardware through clever optimizations.

George's blog post ["Can tinygrad win?"](https://geohot.github.io//blog/jekyll/update/2025/07/06/can-tinygrad-win.html) made eve

---
### assistant

```
context[2000:3000]
```

---
### tool

rything click. He lays out his vision for the library: the entire stack of making code run fast on hardware is fundamentally a search problem across multiple layers — searching for better scheduling algorithms, faster GPU kernels, all of it. The task of the library is to "Expose the underlying search problem spanning several orders of magnitude." and then "Apply the state of the art in search. Burn compute.".

That's when I finally understood what Sutton meant. Like in computer chess, raw compute and effective search will beat manually crafted solutions.

But the key insight is this: someone still has to expose that search problem. Someone has to frame it correctly.

In this post, I'll show you why our work as engineers still matters, just differently. The Bitter Lesson isn't bitter at all - it's a blueprint for better engineering. And the pattern of search & compute appears everywhere once you know what to look for.

## The Framework

For me, The Bitter Lesson is all about building ef

---
### assistant

```
context[3000:4000]
```

---
### tool

fective search systems. Again, not search in the sense of information retrieval, but in the sense of allowing raw compute to explore a space of solutions effectively.

For that we need three core components:
1. An effective problem formulation that exposes the right parameters in a way that different algorithms can dig into it.
2. A way to evaluate possible solutions (evals) and provide feedback to the search mechanism.
3. A way to define constraints that a solution has to satisfy (memory, time, specific form etc.)

Our job as engineers is to build these three components. We design the problem-specific, scalable infrastructure that enables search to work. Then we gtfo and let compute do what it does best.

But don't you worry, there is still place for your precious domain knowledge, even in The Bitter Lesson, but you should use it elsewhere:

Use your domain knowledge to:

* Design problem representations (make search tractable)
* Build evaluations (capture what matters)
* Set constrai

---
### assistant

```
context[4000:5000]
```

---
### tool

nts (avoid unsafe regions)
* Initialize search (warm start)

And use search to:

* Explore solution space
* Find non-obvious patterns
* Exceed human intuition
* Scale the solution with compute

Sound familiar? We've done this before — moving from punch cards to text-based code was the same kind of shift. And now we're building harnesses and guardrails (and [RL environments](https://www.primeintellect.ai/blog/environments)) that define the bounds of desired solutions rather than crafting the solutions directly.

## The Examples

It's fascinating to see where these themes come up, besides `tinygrad`.

If you follow me on [Twitter](https://twitter.com/rasmus1610), you might know that I'm a huge fanboi of DSPy - a declarative library for "programming not prompting" LLMs. And DSPy embodies the approach described above by separating problem definition from solution strategy. Omar Khattab, the creator of the library, actually gave a talk called ["On Engineering AI Systems that Endure The Bitt

---
### assistant

```
context[5000:6000]
```

---
### tool

er Lesson"](https://youtu.be/qdmxApz3EJI?si=P8CFiY1gD1lCMFoG) at this year's AI Engineering conference in San Francisco. Interestingly, his vision of DSPy mirrors George Hotz's vision for tinygrad: provide the right level of abstraction, then let search & compute do its thing.

One key mechanism here is [DSPy's signatures](https://dspy.ai/learn/programming/signatures/). A DSPy signature is like a function signature in the sense that you specify your inputs and the shape of your desired outputs. It gives you the flexibility to go from input to output however you want. Sure, currently this "however" means LLMs, but just because they are the most flexible piece of computing we currently have. But if some other paradigm (or even a newer LLM) comes around the corner, your DSPy modules are very easy to adapt to it.

This search & compute principle is also captured beautifully in a talk I stumbled upon some weeks ago. [Rens Dimmendaal](https://rensdimmendaal.com/) on the [Solveit Discord serv

---
### assistant

```
context[6000:7000]
```

---
### tool

er](https://solve.it.com) pointed me to [a talk by Raymond Hettinger](https://www.youtube.com/watch?v=_GP9OpZPUYc) aptly titled "Modern solvers: Problems well-defined are problems solved". Just the title captures perfectly what I think The Bitter Lesson is all about.

One of the topics in his talk, SAT solvers, shows how problem formulation may become the critical engineering work. For those who don't know what a [SAT solver](https://en.wikipedia.org/wiki/SAT_solver) is, go and read the Wikipedia page. I'm kidding. It's a piece of software which can solve boolean algebra problems very effectively - and you would be surprised how many problems can be formulated as boolean algebra problems. But - without going into too much technical detail (you can really read the wikipedia page - it's linked above) - it expects the input to be in a very specific logical form, called the [conjunctive normal form (CNF)](https://en.wikipedia.org/wiki/Conjunctive_normal_form). And it turns out that just de

---
### assistant

```
context[7000:8000]
```

---
### tool

fining your problem in CNF is often surprisingly tricky. But once you got there, a modern SAT solver gives you the solution very fast, even for very complex problems.

It's also no coincidence that [AI evals](https://maven.com/parlance-labs/evals) have become such a hot topic for building LLM apps recently (h/t [Hamel Husain](https://hamel.dev)). When you're iterating on LLM applications - whether manually or with prompt optimizers - you're essentially searching through a solution space. But unlike a binary classifier where you can rely on simple metrics like accuracy, LLMs have massive output spaces that are much harder to evaluate. Here, building robust evals becomes critical engineering work: you need to define what "good" means for your specific problem, build systems to measure it reliably, and scale that evaluation. This is another instance of building the harness — the infrastructure that lets search & compute work effectively.

But this principle even extends to the physical wo

---
### assistant

```
context[8000:9000]
```

---
### tool

rld. [Proxima Fusion](https://www.youtube.com/watch?v=3Nr__6DDk6I) optimized their stellarator fusion reactor design through numerical simulation—defining the physics constraints, then letting compute search the design space before building the hardware. [AlphaFold](https://alphafold.ebi.ac.uk/) does something similar for proteins: define the structure prediction problem, set up physical constraints, then let search explore conformational space.

## The Conclusion

Look, maybe I'm fooling myself here. Maybe Sutton is right all the way down, and eventually even problem formulation gets automated. Maybe we'll all end up sipping synthetic mojitos in the metaverse while AGI does everything. But we're clearly not there yet - and I don't think we will be anytime soon.

So for now, when I think of The Bitter Lesson, I'm not depressed because we software engineers will become obsolete, but I'm excited because it points to what we should focus on: Building scalable systems that leverage search 

---
### user

You have used all your tool calls for this turn. Please summarize your findings. If you did not complete your goal, tell the user what further work is needed. You may use tools again on the next user message.

---
### assistant

I've analyzed the context of the blog post titled "The Not-so Bitter Lesson." Here are the three key take-home messages from the post:

1. **Effective Problem Formulation**: Engineers need to focus on formulating problems in a way that exposes the right parameters for algorithms to explore. This involves designing problem representations that make search tractable.

2. **Building Evaluation Mechanisms**: It's crucial to create robust evaluation systems that can measure the effectiveness of solutions. This allows the search mechanism to receive feedback and refine its approach.

3. **Defining Constraints**: Engineers should set constraints that solutions must satisfy, such as memory and time limits. This helps guide the search process and ensures that the solutions are practical and safe.

The overall message emphasizes that while raw compute and search can outperform domain-specific solutions, the role of engineers is still vital in framing problems, building evaluations, and defining constraints to enable effective search and compute. 

If you need further details or specific examples from the blog post, please let me know!

---

</div>

Let's print the tool output in code brackets as well and add emojis to the role title thingy

There may be messages without content (content=None) and we need to make sure that our function does not crash, so we have to add a check here.

In [ ]:
#| eval: false
print_history(rlm)

<div class="prose">

### user

what were the 3 take at home messages in the blogpost

---
### assistant

```
len(context)
```

---
### tool

9245

---
### assistant

```
context[:1000]
```

---
### tool

---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tinygrad helped me finally understand what Sutton meant
---

Have you heard about The Bitter Lesson? If you are on Twitter and follow some ML/AI people, I bet you have.

If you haven't: It is a [blog post by Richard Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html), a Canadian computer scientist, who wrote one of *the* books about reinforcement learning.

Sutton's core argument is simple: general methods that leverage search and compute outperform domain-specific solutions. By "search," he doesn't mean information retrieval - he means using learning algorithms to explore possible solutions or strategies. His go-to example is computer chess: early engines relied heavily on chess expertise to evaluate positions, while AlphaZero knew only the basic rules and learned by playing itself, eventually surpassing any human player.

This lesson is "bitter" because it suggests our cleve

---
### assistant

```
context[1000:2000]
```

---
### tool

r insights don't matter. Our domain-specific knowledge gets crushed by what looks like "dumb brute-force search and compute." If you're an engineer, this naturally raises an uncomfortable question: what's the point of us?

## The Click

For a long time, I didn't really understand Sutton's point. What does it mean to solve problems using search and compute? What is the role of engineers then? Are we just supposed to prompt a LLM and hope for the best? That can't scale.

My aha moment came from an unlikely source: [`tinygrad`](https://github.com/tinygrad/tinygrad). It's a Python library similar to PyTorch but much simpler on the surface. I think George Hotz initially built it as a toy project after watching Andrej Karpathy create micrograd, but it evolved into a solid tensor library that runs LLMs efficiently on different hardware through clever optimizations.

George's blog post ["Can tinygrad win?"](https://geohot.github.io//blog/jekyll/update/2025/07/06/can-tinygrad-win.html) made eve

---
### assistant

```
context[2000:3000]
```

---
### tool

rything click. He lays out his vision for the library: the entire stack of making code run fast on hardware is fundamentally a search problem across multiple layers — searching for better scheduling algorithms, faster GPU kernels, all of it. The task of the library is to "Expose the underlying search problem spanning several orders of magnitude." and then "Apply the state of the art in search. Burn compute.".

That's when I finally understood what Sutton meant. Like in computer chess, raw compute and effective search will beat manually crafted solutions.

But the key insight is this: someone still has to expose that search problem. Someone has to frame it correctly.

In this post, I'll show you why our work as engineers still matters, just differently. The Bitter Lesson isn't bitter at all - it's a blueprint for better engineering. And the pattern of search & compute appears everywhere once you know what to look for.

## The Framework

For me, The Bitter Lesson is all about building ef

---
### assistant

```
context[3000:4000]
```

---
### tool

fective search systems. Again, not search in the sense of information retrieval, but in the sense of allowing raw compute to explore a space of solutions effectively.

For that we need three core components:
1. An effective problem formulation that exposes the right parameters in a way that different algorithms can dig into it.
2. A way to evaluate possible solutions (evals) and provide feedback to the search mechanism.
3. A way to define constraints that a solution has to satisfy (memory, time, specific form etc.)

Our job as engineers is to build these three components. We design the problem-specific, scalable infrastructure that enables search to work. Then we gtfo and let compute do what it does best.

But don't you worry, there is still place for your precious domain knowledge, even in The Bitter Lesson, but you should use it elsewhere:

Use your domain knowledge to:

* Design problem representations (make search tractable)
* Build evaluations (capture what matters)
* Set constrai

---
### assistant

```
context[4000:5000]
```

---
### tool

nts (avoid unsafe regions)
* Initialize search (warm start)

And use search to:

* Explore solution space
* Find non-obvious patterns
* Exceed human intuition
* Scale the solution with compute

Sound familiar? We've done this before — moving from punch cards to text-based code was the same kind of shift. And now we're building harnesses and guardrails (and [RL environments](https://www.primeintellect.ai/blog/environments)) that define the bounds of desired solutions rather than crafting the solutions directly.

## The Examples

It's fascinating to see where these themes come up, besides `tinygrad`.

If you follow me on [Twitter](https://twitter.com/rasmus1610), you might know that I'm a huge fanboi of DSPy - a declarative library for "programming not prompting" LLMs. And DSPy embodies the approach described above by separating problem definition from solution strategy. Omar Khattab, the creator of the library, actually gave a talk called ["On Engineering AI Systems that Endure The Bitt

---
### assistant

```
context[5000:6000]
```

---
### tool

er Lesson"](https://youtu.be/qdmxApz3EJI?si=P8CFiY1gD1lCMFoG) at this year's AI Engineering conference in San Francisco. Interestingly, his vision of DSPy mirrors George Hotz's vision for tinygrad: provide the right level of abstraction, then let search & compute do its thing.

One key mechanism here is [DSPy's signatures](https://dspy.ai/learn/programming/signatures/). A DSPy signature is like a function signature in the sense that you specify your inputs and the shape of your desired outputs. It gives you the flexibility to go from input to output however you want. Sure, currently this "however" means LLMs, but just because they are the most flexible piece of computing we currently have. But if some other paradigm (or even a newer LLM) comes around the corner, your DSPy modules are very easy to adapt to it.

This search & compute principle is also captured beautifully in a talk I stumbled upon some weeks ago. [Rens Dimmendaal](https://rensdimmendaal.com/) on the [Solveit Discord serv

---
### assistant

```
context[6000:7000]
```

---
### tool

er](https://solve.it.com) pointed me to [a talk by Raymond Hettinger](https://www.youtube.com/watch?v=_GP9OpZPUYc) aptly titled "Modern solvers: Problems well-defined are problems solved". Just the title captures perfectly what I think The Bitter Lesson is all about.

One of the topics in his talk, SAT solvers, shows how problem formulation may become the critical engineering work. For those who don't know what a [SAT solver](https://en.wikipedia.org/wiki/SAT_solver) is, go and read the Wikipedia page. I'm kidding. It's a piece of software which can solve boolean algebra problems very effectively - and you would be surprised how many problems can be formulated as boolean algebra problems. But - without going into too much technical detail (you can really read the wikipedia page - it's linked above) - it expects the input to be in a very specific logical form, called the [conjunctive normal form (CNF)](https://en.wikipedia.org/wiki/Conjunctive_normal_form). And it turns out that just de

---
### assistant

```
context[7000:8000]
```

---
### tool

fining your problem in CNF is often surprisingly tricky. But once you got there, a modern SAT solver gives you the solution very fast, even for very complex problems.

It's also no coincidence that [AI evals](https://maven.com/parlance-labs/evals) have become such a hot topic for building LLM apps recently (h/t [Hamel Husain](https://hamel.dev)). When you're iterating on LLM applications - whether manually or with prompt optimizers - you're essentially searching through a solution space. But unlike a binary classifier where you can rely on simple metrics like accuracy, LLMs have massive output spaces that are much harder to evaluate. Here, building robust evals becomes critical engineering work: you need to define what "good" means for your specific problem, build systems to measure it reliably, and scale that evaluation. This is another instance of building the harness — the infrastructure that lets search & compute work effectively.

But this principle even extends to the physical wo

---
### assistant

```
context[8000:9000]
```

---
### tool

rld. [Proxima Fusion](https://www.youtube.com/watch?v=3Nr__6DDk6I) optimized their stellarator fusion reactor design through numerical simulation—defining the physics constraints, then letting compute search the design space before building the hardware. [AlphaFold](https://alphafold.ebi.ac.uk/) does something similar for proteins: define the structure prediction problem, set up physical constraints, then let search explore conformational space.

## The Conclusion

Look, maybe I'm fooling myself here. Maybe Sutton is right all the way down, and eventually even problem formulation gets automated. Maybe we'll all end up sipping synthetic mojitos in the metaverse while AGI does everything. But we're clearly not there yet - and I don't think we will be anytime soon.

So for now, when I think of The Bitter Lesson, I'm not depressed because we software engineers will become obsolete, but I'm excited because it points to what we should focus on: Building scalable systems that leverage search 

---
### user

You have used all your tool calls for this turn. Please summarize your findings. If you did not complete your goal, tell the user what further work is needed. You may use tools again on the next user message.

---
### assistant

I've analyzed the context of the blog post titled "The Not-so Bitter Lesson." Here are the three key take-home messages from the post:

1. **Effective Problem Formulation**: Engineers need to focus on formulating problems in a way that exposes the right parameters for algorithms to explore. This involves designing problem representations that make search tractable.

2. **Building Evaluation Mechanisms**: It's crucial to create robust evaluation systems that can measure the effectiveness of solutions. This allows the search mechanism to receive feedback and refine its approach.

3. **Defining Constraints**: Engineers should set constraints that solutions must satisfy, such as memory and time limits. This helps guide the search process and ensures that the solutions are practical and safe.

The overall message emphasizes that while raw compute and search can outperform domain-specific solutions, the role of engineers is still vital in framing problems, building evaluations, and defining constraints to enable effective search and compute. 

If you need further details or specific examples from the blog post, please let me know!

---

</div>

In [ ]:
#| export
def _md(x):
    if isinstance(x, litellm.types.utils.Message): x = x.__dict__

    role = x["role"]

    symbols = {"user": "😀", "assistant": "🤖", "tool": "🧰"}

    md = ""

    md += f"### {symbols[role]} {role}\n\n"
    if x["content"]:
        if role == "tool":
            md += ("```\n" + x["content"] + "\n```\n\n")
        else:
            md += (x["content"] + "\n\n")

    if "tool_calls" in x and x["tool_calls"]:
        c = json.loads(x["tool_calls"][0]["function"]["arguments"])["python_code"]
        md += f"```\n{c}\n```\n\n"

    md+= "---"

    return md

In [ ]:
#| eval: false
Markdown("\n".join(map(_md, rlm.chat.hist)))

<div class="prose">

### 😀 user

what were the 3 take at home messages in the blogpost

---
### 🤖 assistant

```
len(context)
```

---
### 🧰 tool

```
9245
```

---
### 🤖 assistant

```
context[:1000]
```

---
### 🧰 tool

```
---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tinygrad helped me finally understand what Sutton meant
---

Have you heard about The Bitter Lesson? If you are on Twitter and follow some ML/AI people, I bet you have.

If you haven't: It is a [blog post by Richard Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html), a Canadian computer scientist, who wrote one of *the* books about reinforcement learning.

Sutton's core argument is simple: general methods that leverage search and compute outperform domain-specific solutions. By "search," he doesn't mean information retrieval - he means using learning algorithms to explore possible solutions or strategies. His go-to example is computer chess: early engines relied heavily on chess expertise to evaluate positions, while AlphaZero knew only the basic rules and learned by playing itself, eventually surpassing any human player.

This lesson is "bitter" because it suggests our cleve
```

---
### 🤖 assistant

```
context[1000:2000]
```

---
### 🧰 tool

```
r insights don't matter. Our domain-specific knowledge gets crushed by what looks like "dumb brute-force search and compute." If you're an engineer, this naturally raises an uncomfortable question: what's the point of us?

## The Click

For a long time, I didn't really understand Sutton's point. What does it mean to solve problems using search and compute? What is the role of engineers then? Are we just supposed to prompt a LLM and hope for the best? That can't scale.

My aha moment came from an unlikely source: [`tinygrad`](https://github.com/tinygrad/tinygrad). It's a Python library similar to PyTorch but much simpler on the surface. I think George Hotz initially built it as a toy project after watching Andrej Karpathy create micrograd, but it evolved into a solid tensor library that runs LLMs efficiently on different hardware through clever optimizations.

George's blog post ["Can tinygrad win?"](https://geohot.github.io//blog/jekyll/update/2025/07/06/can-tinygrad-win.html) made eve
```

---
### 🤖 assistant

```
context[2000:3000]
```

---
### 🧰 tool

```
rything click. He lays out his vision for the library: the entire stack of making code run fast on hardware is fundamentally a search problem across multiple layers — searching for better scheduling algorithms, faster GPU kernels, all of it. The task of the library is to "Expose the underlying search problem spanning several orders of magnitude." and then "Apply the state of the art in search. Burn compute.".

That's when I finally understood what Sutton meant. Like in computer chess, raw compute and effective search will beat manually crafted solutions.

But the key insight is this: someone still has to expose that search problem. Someone has to frame it correctly.

In this post, I'll show you why our work as engineers still matters, just differently. The Bitter Lesson isn't bitter at all - it's a blueprint for better engineering. And the pattern of search & compute appears everywhere once you know what to look for.

## The Framework

For me, The Bitter Lesson is all about building ef
```

---
### 🤖 assistant

```
context[3000:4000]
```

---
### 🧰 tool

```
fective search systems. Again, not search in the sense of information retrieval, but in the sense of allowing raw compute to explore a space of solutions effectively.

For that we need three core components:
1. An effective problem formulation that exposes the right parameters in a way that different algorithms can dig into it.
2. A way to evaluate possible solutions (evals) and provide feedback to the search mechanism.
3. A way to define constraints that a solution has to satisfy (memory, time, specific form etc.)

Our job as engineers is to build these three components. We design the problem-specific, scalable infrastructure that enables search to work. Then we gtfo and let compute do what it does best.

But don't you worry, there is still place for your precious domain knowledge, even in The Bitter Lesson, but you should use it elsewhere:

Use your domain knowledge to:

* Design problem representations (make search tractable)
* Build evaluations (capture what matters)
* Set constrai
```

---
### 🤖 assistant

```
context[4000:5000]
```

---
### 🧰 tool

```
nts (avoid unsafe regions)
* Initialize search (warm start)

And use search to:

* Explore solution space
* Find non-obvious patterns
* Exceed human intuition
* Scale the solution with compute

Sound familiar? We've done this before — moving from punch cards to text-based code was the same kind of shift. And now we're building harnesses and guardrails (and [RL environments](https://www.primeintellect.ai/blog/environments)) that define the bounds of desired solutions rather than crafting the solutions directly.

## The Examples

It's fascinating to see where these themes come up, besides `tinygrad`.

If you follow me on [Twitter](https://twitter.com/rasmus1610), you might know that I'm a huge fanboi of DSPy - a declarative library for "programming not prompting" LLMs. And DSPy embodies the approach described above by separating problem definition from solution strategy. Omar Khattab, the creator of the library, actually gave a talk called ["On Engineering AI Systems that Endure The Bitt
```

---
### 🤖 assistant

```
context[5000:6000]
```

---
### 🧰 tool

```
er Lesson"](https://youtu.be/qdmxApz3EJI?si=P8CFiY1gD1lCMFoG) at this year's AI Engineering conference in San Francisco. Interestingly, his vision of DSPy mirrors George Hotz's vision for tinygrad: provide the right level of abstraction, then let search & compute do its thing.

One key mechanism here is [DSPy's signatures](https://dspy.ai/learn/programming/signatures/). A DSPy signature is like a function signature in the sense that you specify your inputs and the shape of your desired outputs. It gives you the flexibility to go from input to output however you want. Sure, currently this "however" means LLMs, but just because they are the most flexible piece of computing we currently have. But if some other paradigm (or even a newer LLM) comes around the corner, your DSPy modules are very easy to adapt to it.

This search & compute principle is also captured beautifully in a talk I stumbled upon some weeks ago. [Rens Dimmendaal](https://rensdimmendaal.com/) on the [Solveit Discord serv
```

---
### 🤖 assistant

```
context[6000:7000]
```

---
### 🧰 tool

```
er](https://solve.it.com) pointed me to [a talk by Raymond Hettinger](https://www.youtube.com/watch?v=_GP9OpZPUYc) aptly titled "Modern solvers: Problems well-defined are problems solved". Just the title captures perfectly what I think The Bitter Lesson is all about.

One of the topics in his talk, SAT solvers, shows how problem formulation may become the critical engineering work. For those who don't know what a [SAT solver](https://en.wikipedia.org/wiki/SAT_solver) is, go and read the Wikipedia page. I'm kidding. It's a piece of software which can solve boolean algebra problems very effectively - and you would be surprised how many problems can be formulated as boolean algebra problems. But - without going into too much technical detail (you can really read the wikipedia page - it's linked above) - it expects the input to be in a very specific logical form, called the [conjunctive normal form (CNF)](https://en.wikipedia.org/wiki/Conjunctive_normal_form). And it turns out that just de
```

---
### 🤖 assistant

```
context[7000:8000]
```

---
### 🧰 tool

```
fining your problem in CNF is often surprisingly tricky. But once you got there, a modern SAT solver gives you the solution very fast, even for very complex problems.

It's also no coincidence that [AI evals](https://maven.com/parlance-labs/evals) have become such a hot topic for building LLM apps recently (h/t [Hamel Husain](https://hamel.dev)). When you're iterating on LLM applications - whether manually or with prompt optimizers - you're essentially searching through a solution space. But unlike a binary classifier where you can rely on simple metrics like accuracy, LLMs have massive output spaces that are much harder to evaluate. Here, building robust evals becomes critical engineering work: you need to define what "good" means for your specific problem, build systems to measure it reliably, and scale that evaluation. This is another instance of building the harness — the infrastructure that lets search & compute work effectively.

But this principle even extends to the physical wo
```

---
### 🤖 assistant

```
context[8000:9000]
```

---
### 🧰 tool

```
rld. [Proxima Fusion](https://www.youtube.com/watch?v=3Nr__6DDk6I) optimized their stellarator fusion reactor design through numerical simulation—defining the physics constraints, then letting compute search the design space before building the hardware. [AlphaFold](https://alphafold.ebi.ac.uk/) does something similar for proteins: define the structure prediction problem, set up physical constraints, then let search explore conformational space.

## The Conclusion

Look, maybe I'm fooling myself here. Maybe Sutton is right all the way down, and eventually even problem formulation gets automated. Maybe we'll all end up sipping synthetic mojitos in the metaverse while AGI does everything. But we're clearly not there yet - and I don't think we will be anytime soon.

So for now, when I think of The Bitter Lesson, I'm not depressed because we software engineers will become obsolete, but I'm excited because it points to what we should focus on: Building scalable systems that leverage search 
```

---
### 😀 user

You have used all your tool calls for this turn. Please summarize your findings. If you did not complete your goal, tell the user what further work is needed. You may use tools again on the next user message.

---
### 🤖 assistant

I've analyzed the context of the blog post titled "The Not-so Bitter Lesson." Here are the three key take-home messages from the post:

1. **Effective Problem Formulation**: Engineers need to focus on formulating problems in a way that exposes the right parameters for algorithms to explore. This involves designing problem representations that make search tractable.

2. **Building Evaluation Mechanisms**: It's crucial to create robust evaluation systems that can measure the effectiveness of solutions. This allows the search mechanism to receive feedback and refine its approach.

3. **Defining Constraints**: Engineers should set constraints that solutions must satisfy, such as memory and time limits. This helps guide the search process and ensures that the solutions are practical and safe.

The overall message emphasizes that while raw compute and search can outperform domain-specific solutions, the role of engineers is still vital in framing problems, building evaluations, and defining constraints to enable effective search and compute. 

If you need further details or specific examples from the blog post, please let me know!

---

</div>

the markdown function helps with making tracing rather easier to review. How can we incorporate it into the main class?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

You could add a `trace()` method to the `RLM` class that returns the formatted markdown. Something like:

```python
def trace(self):
    return Markdown("\n".join(map(_md, self.chat.hist)))
```

Then you'd just call `rlm.trace()` after a query to see the pretty-printed trajectory.

Want to also keep `_md` as a static method inside the class, or leave it as a standalone function?

In [ ]:
#| eval: false
rlm = RLM(context=bp, root_model="openrouter/openai/gpt-4o-mini", temperature=0.2)
rlm("give me the synopsis of the blogpost")

The context provided is a blog post titled "The Not-so Bitter Lesson," dated October 20, 2025. It discusses Richard Sutton's concept known as "The Bitter Lesson," which emphasizes that general methods leveraging search and computation tend to outperform domain-specific solutions in machine learning and artificial intelligence. The post uses the example of computer chess, contrasting early engines that relied on chess expertise with AlphaZero, which learned solely through self-play and ultimately surpassed human players.

To provide a complete synopsis, I will need to analyze more of the context to capture the full scope of the blog post. I can continue to extract and summarize additional sections in the next step.

<details>

- id: `gen-1774863243-R7XUk26qLHuDiyYrhgbV`
- model: `openai/gpt-4o-mini`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=131, prompt_tokens=1166, total_tokens=1297, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=0.0002535, is_byok=False, cost_details={'upstream_inference_cost': 0.0002535, 'upstream_inference_prompt_cost': 0.0001749, 'upstream_inference_completions_cost': 7.86e-05})`

</details>

In [ ]:
#| eval: false
rlm.trace()

<div class="prose">

### 😀 user

give me the synopsis of the blogpost

---
### 🤖 assistant

```
len(context)
```

---
### 🧰 tool

```
9245
```

---
### 🤖 assistant

```
context[:1000]
```

---
### 🧰 tool

```
---
title: "The Not-so Bitter Lesson"
date: 2025-10-20T15:27:52+01:00
draft: false
excerpt: How tinygrad helped me finally understand what Sutton meant
---

Have you heard about The Bitter Lesson? If you are on Twitter and follow some ML/AI people, I bet you have.

If you haven't: It is a [blog post by Richard Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html), a Canadian computer scientist, who wrote one of *the* books about reinforcement learning.

Sutton's core argument is simple: general methods that leverage search and compute outperform domain-specific solutions. By "search," he doesn't mean information retrieval - he means using learning algorithms to explore possible solutions or strategies. His go-to example is computer chess: early engines relied heavily on chess expertise to evaluate positions, while AlphaZero knew only the basic rules and learned by playing itself, eventually surpassing any human player.

This lesson is "bitter" because it suggests our cleve
```

---
### 😀 user

You have used all your tool calls for this turn. Please summarize your findings. If you did not complete your goal, tell the user what further work is needed. You may use tools again on the next user message.

---
### 🤖 assistant

The context provided is a blog post titled "The Not-so Bitter Lesson," dated October 20, 2025. It discusses Richard Sutton's concept known as "The Bitter Lesson," which emphasizes that general methods leveraging search and computation tend to outperform domain-specific solutions in machine learning and artificial intelligence. The post uses the example of computer chess, contrasting early engines that relied on chess expertise with AlphaZero, which learned solely through self-play and ultimately surpassed human players.

To provide a complete synopsis, I will need to analyze more of the context to capture the full scope of the blog post. I can continue to extract and summarize additional sections in the next step.

---

</div>

## Wrap up

Please look at my implementation and the original blog post and see if my implementation is faithful to the original blog post.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let me re-read the blog post to do a careful comparison.


<details class='tool-usage-details'>
<summary><code>read_url(url=&#x27;https://alexzhang13.github.io/blog/2025/…&#x27;, selector=&#x27;.content&#x27;)→&#x27;No output&#x27;</code></summary>

```json
{
  "id": "toolu_017KkvCXUfYrTv2VRwJpHQb1",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://alexzhang13.github.io/blog/2025/rlm/",
      "selector": ".content"
    }
  },
  "result": "No output"
}
```

</details>




<details class='tool-usage-details'>
<summary><code>read_url(url=&#x27;https://alexzhang13.github.io/blog/2025/…&#x27;)→&#x27;## **Recursive Language Models (RLMs).**…&#x27;</code></summary>

```json
{
  "id": "toolu_01TREu9AeUzj7fjWTnh9Rvpf",
  "call": {
    "function": "read_url",
    "arguments": {
      "url": "https://alexzhang13.github.io/blog/2025/rlm/#recursive-language-models-rlms"
    }
  },
  "result": "<TRUNCATED>\u2026sive Language Models (RLMs).**\n\nA recursive language model is a thin wrapper around a LM that can spawn (recursive) LM calls for intermediate computation \u2014 from the perspective of the user or programmer, it is the same as a model call. In other words, you query a RLM as an \u201cAPI\u201d like you would a LM, i.e. `rlm.completion(messages)` is a direct replacement for `gpt5.completion(messages)`. We take a **context-centric view** rather than a **problem-centric view** of input decomposition. This framing retains the functional view that we want a system that can answer a particular **query** over some associated **context** :\n\n![API](https://alexzhang13.github.io/assets/img/rlm/api.png) **Figure 2.** A recursive language model call replaces a language model call. It provides the user the illusion of near infinite context, while under the hood a language model manages, partitions, and recursively calls itself or another LM over the context accordingly to avoid context rot.\n\nUnder the hood, a RLM provides only the **query** to the LM (which we call the **root LM** , or LM with depth=0), and allows this LM to interact with an **environment** , which stores the (potentially huge) **context**.\n\nWe choose the **environment** to be a loop where the LM can write to and read the output of cells of a Python REPL Notebook (similar to a Jupyter Notebook environment) that is pre-loaded with the **context** as a variable in memory. The **root LM** has the ability to call a recursive LM (or LM with depth=1) inside the REPL **environment** as if it were a function in code, allowing it to naturally peek at, partition, grep through, and launch recursive sub-queries over the **context**. **Figure 3** shows an example of how the RLM with a REPL **environment** produces a final answer.\n\n![API](https://alexzhang13.github.io/assets/img/rlm/repl.png) **Figure 3.** Our instantiation of the RLM framework provides the root LM the ability to analyze the context in a Python notebook environment, and launch recursive LM calls (depth=1) over any string stored in a variable. The LM interacts by outputting code blocks, and it receives a (truncated) version of the output in its context. When it is done, it outputs a final answer with `FINAL(\u2026)` tags or it can choose to use a string in the code execution environment with `FINAL_VAR(\u2026)`.\n\nWhen the **root LM** is confident it has an answer, it can either directly output the answer as `FINAL(answer)`, or it can build up an answer using the variables in its REPL environment, and return the string inside that answer as `FINAL_VAR(final_ans_var)`.\n\nThis setup yields several benefits that are visible in practice:\n\n  1. The context window of the root LM is rarely clogged \u2014 because it never directly sees the entire context, its input context grows slowly.\n  2. The root LM has the flexibility to view subsets of the context, or naively recurse over chunks of it. For example, if the query is to find a needle-in-the-haystack fact or multi-hop fact, the root LM can use `regex` queries to roughly narrow the context, then launch recursive LM calls over this context. This is particularly useful for arbitrary long context inputs, where indexing a retriever is expensive on the fly!\n  3. The context can, in theory, be any modality that can be loaded into memory. The root LM has full control to view and transform this data, as well as ask sub-queries to a recursive LM.\n\n\n\n**Relationship to test-time inference scaling.** We are particularly excited about this view of language models because it offers another axis of scaling test-time compute. The trajectory in which a language model chooses to interact with and recurse over its context is entirely learnable, and can be RL-ified in the same way that reasoning is currently trained for frontier models. Interestingly, it does not directly require training models that can handle huge context lengths because **no single language model call should require handling a huge context**.\n\n**RLMs with REPL environments are powerful.** We highlight that the choice of the **environment** is flexible and not fixed to a REPL or code environment, but we argue that it is a good choice. The two key design choices of recursive language models are 1) treating the prompt as a Python variable, which can be processed programmatically in arbitrary REPL flows. This allows the LLM to figure out what to peek at from the long context, at test time, and to scale any decisions it wants to take (e.g., come up with its own scheme for chunking and recursion adaptively) and 2) allowing that REPL environment to make calls back to the LLM (or a smaller LLM), facilitated by the decomposition and versatility from choice (1).\n\nWe were excited by the design of CodeAct, and reasoned that adding recursive model calls to this system could result in significantly stronger capabilities \u2014 after all, LM function calls are incredibly powerful. However, we argue that RLMs fundamentally view LM usage and\u2026</TRUNCATED>"
}
```

</details>

Good, now let me check the official repo for comparison too:


<details class='tool-usage-details'>
<summary><code>read_gh_repo(owner=&#x27;alexzhang13&#x27;, repo=&#x27;rlm&#x27;)→&#x27;/usr/local/lib/python3.12/site-packages/…&#x27;</code></summary>

```json
{
  "id": "toolu_01CkZ1AxqM1K3rUrvEUhxhRN",
  "call": {
    "function": "read_gh_repo",
    "arguments": {
      "owner": "alexzhang13",
      "repo": "rlm"
    }
  },
  "result": "<TRUNCATED>\u2026/lib/python3.12/site-packages/ghapi/core.py:106: UserWarning: Neither GITHUB_TOKEN nor GITHUB_JWT_TOKEN found: running as unauthenticated\n  else: warn('Neither GITHUB_TOKEN nor GITHUB_JWT_TOKEN found: running as unauthenticated')\n\n'# alexzhang13/rlm\\nGeneral plug-and-play inference library for Recursive Language Models (RLMs), supporting various sandboxes.\\n\\n## / Files\\n- .gitattributes\\n- \ud83d\udcc1 .github\\n- .gitignore\\n- .pre-commit-config.yaml\\n- .python-version\\n- AGENTS.md\\n- CONTRIBUTING.md\\n- LICENSE\\n- MANIFEST.IN\\n- Makefile\\n- README.md\\n- \ud83d\udcc1 docs\\n- \ud83d\udcc1 examples\\n- \ud83d\udcc1 media\\n- pyproject.toml\\n- \ud83d\udcc1 rlm\\n- \ud83d\udcc1 tests\\n- uv.lock\\n- \ud83d\udcc1 visualizer\\n\\n## README\\n\\n---\\n\\n<h1 align=\"center\" style=\"font-size:2.8em\">\\n<span>Recursive Language Models (<span style=\"color:orange\">RLM</span>s)</span>\\n</h1>\\n\\n<p align=\"center\" style=\"font-size:1.3em\">\\n  <a href=\"https://arxiv.org/abs/2512.24601\">Full Paper</a> \u2022\\n  <a href=\"https://alexzhang13.github.io/blog/2025/rlm/\">Blogpost</a> \u2022\\n  <a href=\"https://alexzhang13.github.io/rlm/\">Documentation</a> \u2022\\n  <a href=\"https://github.com/alexzhang13/rlm-minimal\">RLM Minimal</a>\\n</p>\\n\\n<p align=\"center\">\\n  <a href=\"https://github.com/alexzhang13/rlm/actions/workflows/style.yml\">\\n    <img src=\"https://github.com/alexzhang13/rlm/actions/workflows/style.yml/badge.svg\" alt=\"Style\" />\\n  </a>\\n  <a href=\"https://github.com/alexzhang13/rlm/actions/workflows/test.yml\">\\n    <img src=\"https://github.com/alexzhang13/rlm/actions/workflows/test.yml/badge.svg\" alt=\"Test\" />\\n  </a>\\n</p>\\n\\n<p align=\"center\">\\n  <a href=\"https://arxiv.org/abs/2512.24601\">\\n    <img src=\"media/paper_preview.png\" alt=\"Paper Preview\" width=\"300\"/>\\n  </a>\\n</p>\\n\\n## Overview\\nRecursive Language Models (RLMs) are a task-agnostic inference paradigm for language models (LMs) to handle near-infinite length contexts by enabling the LM to *programmatically* examine, decompose, and recursively call itself over its input. RLMs replace the canonical `llm.completion(prompt, model)` call with a `rlm.completion(prompt, model)` call. RLMs offload the context as a variable in a REPL environment that the LM can interact with and launch sub-LM calls inside of.\\n\\nThis repository provides an extensible inference engine for using RLMs around standard API-based and local LLMs. The initial experiments and idea were proposed in a [blogpost](https://alexzhang13.github.io/blog/2025/rlm/) in 2025, with expanded results in an [arXiv preprint](https://arxiv.org/abs/2512.24601).\\n\\n> [!NOTE]\\n> This repository contains inference code for RLMs with support for various sandbox environments. Open-source contributions are welcome. This repository is maintained by the authors of the paper from the MIT OASYS lab.\\n\\n## Quick Setup\\nYou can try out RLMs quickly by installing from PyPi:\\n```bash\\npip install rlms\\n```\\n\\nThe default RLM client uses a REPL environment that runs on the host process through Python `exec` calls. It uses the same virtual environment as the host process (i.e. it will have access to the same dependencies), but with some limitations in its available global modules. As an example, we can call RLM completions using GPT-5-nano:\\n```python\\nfrom rlm import RLM\\n\\nrlm = RLM(\\n    backend=\"openai\",\\n    backend_kwargs={\"model_name\": \"gpt-5-nano\"},\\n    verbose=True,  # For printing to console with rich, disabled by default.\\n)\\n\\nprint(rlm.completion(\"Print me the first 100 powers of two, each on a newline.\").response)\\n```\\n\\n<details>\\n<summary><b>Manual Setup</b></summary>\\n\\nSet up the dependencies with `uv` (or your virtual environment of choice):\\n```bash\\ncurl -LsSf https://astral.sh/uv/install.sh | sh\\nuv init && uv venv --python 3.12  # change version as needed\\nuv pip install -e .\\n```\\n\\nThis project includes a `Makefile` to simplify common tasks.\\n\\n- `make install`: Install base dependencies.\\n- `make check`: Run linter, formatter, and tests.\\n\\nTo run a quick test, the following will run an RLM query with the OpenAI client using your environment variable `OPENAI_API_KEY` (feel free to change this). This will generate console output as well as a log which you can use with the visualizer to explore the trajectories.\\n```bash\\nmake quickstart\\n```\\n\\n</details>\\n\\n## REPL Environments\\nWe support two types of REPL environments -- isolated, and non-isolated. Non-isolated environments (default) run code execution on the same machine as the RLM (e.g. through `exec`), which is pretty reasonable for some local low-risk tasks, like simple benchmarking, but can be problematic if the prompts or tool calls can interact with malicious users. Fully isolated environments use cloud-based sandboxes (e.g. Prime Sandboxes, [Modal Sandboxes](https://modal.com/docs/guide/sandboxes)) to run code generated by the RLM, ensuring complete isolation from the host process. Environments can be added, but we natively support the following: `local` (default), `docker`, `modal`, `prime`, `daytona`, `e2b`.\\n\\n```python\\\u2026</TRUNCATED>"
}
```

</details>





Here's my comparison of your implementation vs the original:

**✅ What you've got right (core architecture):**
- Context stored as a variable in the REPL, not passed to the root LM
- Root LM interacts with the REPL via tool calling (`run_repl`)
- `llm_query()` available inside the REPL for recursive sub-calls
- Truncated REPL outputs to avoid clogging the root LM's context
- Separate root and recursive model support

**⚠️ Missing pieces:**

1. **No `FINAL()` / `FINAL_VAR()` mechanism** — The original has the root LM explicitly signal when it's done via `FINAL(answer)` or `FINAL_VAR(variable_name)`. Your version relies on lisette's `max_steps` to stop, which is why you sometimes see the model get cut off mid-analysis.

2. **No isolated sub-call environments** — In the original, each recursive `llm_query` spawns an *isolated* sub-RLM with its own environment. Yours is a flat `litellm.completion` call (depth=1 only, which is what the paper tested, so this is fine for now).

3. **No logging/trajectory metadata** — The official repo has an `RLMLogger` that writes JSONL for their visualizer. Your `trace()` is a good start but lighter.

4. **No sandboxing** — The official repo supports Docker, Modal, etc. for safe code execution. You're using `toolslm`'s shell directly (fine for local experimentation).

The `FINAL()` mechanism is probably the most impactful missing piece — want to tackle that next?

I think the next step would be to add it to my github with documentation and maybe an example for deep research that allows search over multiple blogposts with tracing?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great plan! To get this ready for GitHub, I'd suggest tackling these in order:

1. **Package structure** — Organize into a proper Python package (e.g. `rlm_lite/`)
2. **README** — With the mermaid diagram, install instructions, and a quick example
3. **Deep research example** — Multiple blog posts as context, with tracing output

Which one do you want to start with?

I have never used nbdev but would you guide me on how I would add that here for package organization?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

nbdev is a great choice — it lets you develop your library in notebooks, and it auto-generates the Python package, docs, and tests from them.

The basic idea is:
1. **You write everything in notebooks** (in a `nbs/` folder)
2. **Cells you mark with `#| export`** become your library code
3. **`nbdev_export`** generates the `.py` files into your package
4. **`nbdev_docs`** generates a documentation site from those same notebooks

To get started, first install it:

```bash
pip install nbdev
```

Then initialize your project:

```bash
nbdev_new
```

This will ask you a few questions (repo name, description, etc.) and create the folder structure for you.

Have you got a repo name in mind, and shall we run through the setup together?

rlm_lite

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Let's set it up! First, install nbdev:

```bash
pip install nbdev
```

Then in your terminal, create a new directory and initialize:

```bash
mkdir rlm_lite && cd rlm_lite
nbdev_new
```

It'll ask you for:
- **Repo name**: `rlm_lite`
- **Description**: something like "A lightweight Recursive Language Model implementation"
- **Author**: your name

Have you run that, or shall we do it from a code cell here?

In [ ]:
#| eval: false
pip install nbdev

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#| eval: false
!mkdir rlm_lite && cd rlm_lite
!nbdev_new

/bin/bash: line 1: nbdev_new: command not found


## Needle In Haystack

Just for funsiees lets see how the implementation works on a needle in haystack type of problem. I will just "borrow" the code from the original repo to generate the problem

In [ ]:
#| eval: false
import random

In [ ]:
#| export
def generate_massive_context(num_lines: int = 1_000_000, answer: str = "1298418") -> str:
    print("Generating massive context with 1M lines...")
    
    # Set of random words to use
    random_words = ["blah", "random", "text", "data", "content", "information", "sample"]
    
    lines = []
    for _ in range(num_lines):
        num_words = random.randint(3, 8)
        line_words = [random.choice(random_words) for _ in range(num_words)]
        lines.append(" ".join(line_words))
    
    # Insert the magic number at a random position (somewhere in the middle)
    low = min(400000, max(0, num_lines - 1))
    high = min(600000, max(0, num_lines - 1))
    magic_position = random.randint(low, high)
    lines[magic_position] = f"The magic number is {answer}"
    
    print(f"Magic number inserted at position {magic_position}")
    
    return "\n".join(lines)

In [ ]:
#| eval: false
ctx = generate_massive_context()

Generating massive context with 1M lines...


Magic number inserted at position 596202


In [ ]:
#| eval: false
rlm = RLM(context=ctx)

rlm("I'm looking for a magic number. What is it?")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************dab8. You can find your API key at https://platform.openai.com/account/api-keys.

In [ ]:
#| eval: false
print_history(rlm)

<div class="prose">

### 😀 user

I'm looking for a magic number. What is it?

---
### 🤖 assistant

---

</div>

Okay this is trivial to solve with regex :D I guess the real deal will be if it can solve multihop problems in massive contexts. 

In [ ]:
#| export
class RLM:
    def __init__(
        self,
        context: str,
        root_model="openai/gpt-oss-120b",
        recursive_model=None,
        temperature=0.0,
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        extra_headers=None,
    ):
        self.context = context
        self.root_model = root_model
        self.recursive_model = recursive_model if recursive_model else root_model
        self.base_url = base_url
        self.api_key = api_key
        self.extra_headers = extra_headers
        self.shell = self._prep_shell()

        def run_repl(python_code: str) -> str:
            "Execute Python code in the REPL environment"
            result = self.shell.run_cell(python_code)
            if result.error_in_exec or result.error_before_exec:
                return f"Error: {result.error_in_exec or result.error_before_exec}"
            output = result.stdout or ""
            if result.result is not None:
                output += str(result.result)
            return output

        self.chat = Chat(
            model=self.root_model,
            tools=[run_repl],
            sp=REPL_SYSTEM_PROMPT,
            temp=temperature,
            api_key=self.api_key,
            #base_url=self.base_url,
            extra_headers=self.extra_headers,
        )

    def _prep_shell(self):
        def llm_query(query: str):
            msg = mk_msg(query)
            return litellm.completion(
                self.recursive_model,
                [msg],
                api_key=self.api_key,
                api_base=self.base_url,
                extra_headers=self.extra_headers,
            ).choices[0].message.content

        sh = get_shell()
        sh.user_ns["context"] = self.context
        sh.user_ns["llm_query"] = llm_query
        return sh

    def __call__(self, query: str, max_steps=2):
        return self.chat(query, max_steps=max_steps)

In [ ]:
#| eval: false
?build_openrouter_rlm

```python
def build_openrouter_rlm(
    context:str, root_model:str='openrouter/gpt-4o-mini', recursive_model:Optional=None, temperature:float=0.0,
    base_url:Optional=None, api_key:Optional=None, extra_headers:Optional=None
):

```



```
Return helpers that behave like `RLM` but explicitly target OpenRouter.
```



**File:** `/tmp/ipykernel_396/1123723355.py`

**Type:** function

In [ ]:
#| eval: false
rlm = build_openrouter_rlm(context=bp, root_model="openai/gpt-4o-mini:nitro")
rlm("give me the synopsis of the blogpost")

AssertionError: Docstring missing!

In [ ]:
#| eval: false
rlm = build_openrouter_rlm(context=bp, root_model="openai/gpt-4o-mini:nitro")

AssertionError: Docstring missing!

seems that I am missing the something?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes, the error says "Docstring missing!" - your `run_repl` function inside `build_openrouter_rlm` needs a docstring because `lisette` uses it to generate the tool schema.

Add a docstring to the function:

```python
def run_repl(python_code: str) -> str:
    "Execute Python code in the REPL environment"
    result = shell.run_cell(python_code)
    ...
```

That should fix it!

this function is missing the Chat 

In [ ]:
#| eval: false
rlm = RLM(context=massive_context)

rlm("I'm looking for a magic number. What is it?")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************dab8. You can find your API key at https://platform.openai.com/account/api-keys.

In [ ]:
#| eval: false
rlm = RLM(context=massive_context)
rlm("where is the world needle")


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers


Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers



AuthenticationError: litellm.AuthenticationError: AuthenticationError: OpenAIException - Incorrect API key provided: sk-or-v1*************************************************************dab8. You can find your API key at https://platform.openai.com/account/api-keys.

Now we can use the model to find the needle in the haystack?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

The auth error is still happening — the `Chat` in your `RLM` class is missing `api_base`. Looking at your latest `RLM` definition:

```python
self.chat = Chat(
    ...
    api_key=self.api_key,
    #base_url=self.base_url,   # <-- this is commented out!
    ...
)
```

The `base_url` is commented out, so `Chat` isn't routing to OpenRouter. Try uncommenting that line (note: the lisette `Chat` parameter is `api_base`, not `base_url`):

```python
self.chat = Chat(
    ...
    api_key=self.api_key,
    api_base=self.base_url,
    ...
)
```

In [ ]:
#| eval: false
rlm = RLM(context=massive_context)

rlm("I'm looking for a magic number. What is it?")

Message(content=None, role='assistant', tool_calls=None, function_call=None, reasoning_content='We need to find magic number in context. First inspect context variable.', provider_specific_fields={'reasoning': 'We need to find magic number in context. First inspect context variable.', 'reasoning_details': [{'type': 'reasoning.text', 'text': 'We need to find magic number in context. First inspect context variable.', 'format': 'unknown', 'index': 0}], 'refusal': None, 'reasoning_content': 'We need to find magic number in context. First inspect context variable.'})

<details>

- id: `gen-1774515652-AFLkWoqL22ej6NwPAojD`
- model: `openai/gpt-oss-120b`
- finish_reason: `stop`
- usage: `Usage(completion_tokens=43, prompt_tokens=925, total_tokens=968, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=18, rejected_prediction_tokens=None, text_tokens=None, image_tokens=0, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=0, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=0, cache_write_tokens=0), cost=9.873e-05, is_byok=False, cost_details={'upstream_inference_cost': 9.873e-05, 'upstream_inference_prompt_cost': 8.325e-05, 'upstream_inference_completions_cost': 1.548e-05})`

</details>

In [ ]:
#| eval: false
rlm = RLM(context=ctx)

rlm("I'm looking for a magic number. What is it?")

NameError: name 'ctx' is not defined

In [ ]:
#| eval: false
rlm = RLM(context=massive_context, root_model="openrouter/openai/gpt-4o-mini", temperature=0.2)

I think this could help with internet search queries over a long horizon for example deep research and allow tracing. How does that sound as a test?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

That's a great idea! Deep research is exactly the kind of task where RLMs shine — you need to:

1. **Gather** information from multiple web searches
2. **Synthesize** across many documents (which can exceed context windows)
3. **Trace** the reasoning chain (which your `trace()` method already supports)

The key addition you'd need is a `web_search` tool injected into the REPL alongside `llm_query` and `context`. Something like a function that takes a query string and returns search results as text.

Would you want to:
- **A)** Add a `web_search()` function to the REPL namespace (so the root LM can search from within code)
- **B)** Add it as a separate tool alongside `run_repl`
- **C)** Both?

Which approach feels right to you?